# Phase 2. Legal Named Entity Recognition (L-NER)

**Task.** Find legal entities in Indian court judgments and return, for every entity, its exact
**character span** and **label**: `[start_char, end_char, label]`. Scoring is **strict**: an entity
counts only if both boundaries *and* the label match exactly.

**Labels (12):** APP, RESP, A.COUNSEL, R.COUNSEL, JUDGE, WIT, AUTH, COURT, STAT, PREC, DATE, CASENO.

**Requirements (spec §8).**
- Start from our **Phase 1 models**; no other pretrained models or LLM APIs.
- At least **6 substantially different experiments**: at least **5 fine-tuning / adaptation techniques**
  (**LoRMA compulsory**) and at least **1 knowledge-distillation** experiment.
- **All four Phase 1 models** must be involved.

**Data.** `fold1` and `fold2` are labelled (we hold out part of them for local validation);
`fold3` is the test set. Submission window: **7–11 October**.

**Plan.**
1. Data inspection: format, offsets, entity statistics, document lengths.
2. Pipeline: char↔token alignment, sliding windows, labels per token, decoding back to character spans,
   and a local strict-F1 evaluator.
3. A shared token-classification head that works with M1–M4.
4. Experiments: full fine-tuning, LoRA, LoRMA, adapters, BitFit / partial fine-tuning, and KD.

## P2.1 Data download and inspection

We download the L-NER folds and check:
- the **annotation format** and whether `text[start:end]` matches the stored entity text (offset correctness);
- **entity statistics**: counts per label (to judge class imbalance) and typical span lengths;
- **overlapping / nested spans** (if there are none, BIO tagging is enough; otherwise we need span classification);
- **document lengths in our tokenizer's tokens** (to choose the sliding-window size and stride).

In [4]:
# ===== Cell N0: which HF token can download the gated capstone dataset? =====
from kaggle_secrets import UserSecretsClient
from huggingface_hub import HfApi, hf_hub_download
for name in ("HF_TOKEN", "HF_WRITE_TOKEN"):
    try:
        t = UserSecretsClient().get_secret(name).strip()          # .strip(): pasted tokens often carry a newline
        user = HfApi(token=t).whoami()["name"]
        hf_hub_download("Exploration-Lab/CS-781-Capstone", "Phase_2_LNER/Train/labels.txt",
                        repo_type="dataset", local_dir="/kaggle/working/data_p2", token=t)
        print(f"{name}: account {user} -> download OK")
    except Exception as e:
        print(f"{name}: FAILED -> {type(e).__name__}: {str(e)[:120]}")

labels.txt:   0%|          | 0.00/71.0 [00:00<?, ?B/s]

HF_TOKEN: account Utkarsh-1718-1 -> download OK
HF_WRITE_TOKEN: account Utkarsh-1718-1 -> download OK


In [5]:
# ===== Cell N1: download Phase 2 (L-NER) data and inspect format + statistics =====
import os, json, collections
from kaggle_secrets import UserSecretsClient
from huggingface_hub import hf_hub_download
from tokenizers import Tokenizer

# ---- paths + download (small files, a few seconds) --------------------------------------------
WORK = "/kaggle/working"; P2 = f"{WORK}/data_p2"; os.makedirs(P2, exist_ok=True)
hf_tok = UserSecretsClient().get_secret("HF_WRITE_TOKEN")
for f in ["Train/fold1-text.json", "Train/fold1-gold.json", "Train/fold2-text.json",
          "Train/fold2-gold.json", "Train/labels.txt", "Test/fold3-text.json"]:
    hf_hub_download("Exploration-Lab/CS-781-Capstone", f"Phase_2_LNER/{f}", repo_type="dataset",
                    local_dir=P2, token=hf_tok)
D = f"{P2}/Phase_2_LNER"
labels = [l.strip() for l in open(f"{D}/Train/labels.txt") if l.strip()]
print("labels:", labels)

# ---- raw format: one gold entry exactly as stored ------------------------------------------------
gold1 = json.load(open(f"{D}/Train/fold1-gold.json"))
k0 = next(iter(gold1))
print("gold container:", type(gold1).__name__, "| entry type:", type(gold1[k0]).__name__)
print("first annotations of doc", k0, ":", gold1[k0][:3])

# ---- our Phase 1 tokenizer (to measure document lengths in model tokens) ----------------------
tk = Tokenizer.from_file(f"{WORK}/tokenizer/hf_a1_bpe_v2_32k.json")

def stats(name, text_f, gold_f=None):
    """Document lengths; if gold is given: label counts, offset check, overlaps, span lengths."""
    T = json.load(open(text_f)); G = json.load(open(gold_f)) if gold_f else {}
    chars = sorted(map(len, T.values()))
    ntok = sorted(len(tk.encode(t, add_special_tokens=False).ids) for t in T.values())
    print(f"\n[{name}] docs {len(T)} | chars median {chars[len(chars)//2]} | tokens median "
          f"{ntok[len(ntok)//2]}  p90 {ntok[int(.9*len(ntok))]}  max {ntok[-1]}")
    if not G:
        return
    cnt, bad, overlap, lens = collections.Counter(), 0, 0, []
    for d, ents in G.items():
        txt = T[d]
        spans = sorted((e[0], e[1]) for e in ents)
        overlap += sum(spans[i][1] > spans[i + 1][0] for i in range(len(spans) - 1))  # nested/overlapping
        for e in ents:
            s, t, lab = e[0], e[1], e[2]
            cnt[lab] += 1; lens.append(t - s)
            if len(e) > 3 and txt[s:t] != e[3]:
                bad += 1                                                             # offset mismatch
    lens.sort()
    print(f"  entities {sum(cnt.values())} | per doc {sum(cnt.values())/len(G):.1f} | overlapping pairs "
          f"{overlap} | offset mismatches {bad} | span chars median {lens[len(lens)//2]} p95 {lens[int(.95*len(lens))]}")
    print("  per label:", dict(cnt.most_common()))

stats("fold1", f"{D}/Train/fold1-text.json", f"{D}/Train/fold1-gold.json")
stats("fold2", f"{D}/Train/fold2-text.json", f"{D}/Train/fold2-gold.json")
stats("fold3 TEST", f"{D}/Test/fold3-text.json")

fold1-text.json: 0.00B [00:00, ?B/s]

fold1-gold.json: 0.00B [00:00, ?B/s]

fold2-text.json: 0.00B [00:00, ?B/s]

fold2-gold.json: 0.00B [00:00, ?B/s]

fold3-text.json: 0.00B [00:00, ?B/s]

labels: ['APP', 'RESP', 'A.COUNSEL', 'R.COUNSEL', 'JUDGE', 'WIT', 'AUTH', 'COURT', 'STAT', 'PREC', 'DATE', 'CASENO']
gold container: dict | entry type: list
first annotations of doc 115651329 : [[137, 153, 'RESP', 'STATE OF HARYANA'], [252, 261, 'DATE', '19.8.2011'], [276, 308, 'COURT', 'High Court of Punjab and Haryana']]

[fold1] docs 35 | chars median 26600 | tokens median 5565  p90 11410  max 22126
  entities 4120 | per doc 117.7 | overlapping pairs 0 | offset mismatches 0 | span chars median 13 p95 54
  per label: {'DATE': 813, 'PREC': 624, 'AUTH': 535, 'STAT': 525, 'COURT': 455, 'CASENO': 423, 'APP': 207, 'RESP': 191, 'JUDGE': 123, 'R.COUNSEL': 87, 'A.COUNSEL': 70, 'WIT': 67}

[fold2] docs 35 | chars median 26730 | tokens median 5822  p90 8670  max 23918
  entities 4273 | per doc 122.1 | overlapping pairs 0 | offset mismatches 0 | span chars median 13 p95 50
  per label: {'DATE': 762, 'STAT': 735, 'PREC': 559, 'COURT': 558, 'AUTH': 468, 'CASENO': 337, 'APP': 200, 'RESP': 190, 'A.

## P2.2 Pipeline: exact character offsets ↔ model tokens

Our Phase 1 models were trained on **normalised** text (ASCII only, single spaces, no newlines), but
L-NER is scored on **original** character offsets, *strictly*. So `p2_ner.py`:

1. **Normalises with an index map**: `normalise_with_map(text)` returns exactly the Phase 1 normalised
   text *and*, for every normalised character, its index in the original text.
2. **Tokenises** with our A1-BPE tokenizer and maps each token's *core* (leading space removed)
   back to an original span.
3. **BIO tags** (25 = O + B/I × 12 labels): a token is `B-X` if it starts exactly at an entity start,
   `I-X` if it lies inside the entity. No spans overlap, so BIO is sufficient.
4. **Decodes** tags back to `[start, end, label]` in original offsets, repairing stray `I-X`.
5. **Sliding windows** (512 tokens, stride 256) for long documents.
6. **Strict micro-F1** (exact span and label) plus per-label F1, as in the official evaluator.

**Dev split:** 5 documents per fold (10 of 70) are held out, chosen deterministically by a hash of the document id.

**Oracle check:** gold → BIO tags → decoded spans gives the *upper bound* of any token tagger. An entity
whose boundary falls **inside** a token cannot be recovered.

In [6]:
%%writefile /kaggle/working/src/p2_ner.py
"""
p2_ner.py -- Phase 2 (Legal NER) data pipeline: exact character offsets <-> model tokens.

Why this file exists
  Our Phase-1 models were trained on NORMALISED text (ASCII only, single spaces, no newlines),
  but L-NER is scored on character offsets of the ORIGINAL text, strictly (one wrong character
  = wrong entity). So we normalise each document while keeping, for every normalised character,
  the index of the original character it came from. Tokens are then mapped back exactly.

Pipeline
  normalise_with_map(text)          -> norm, idx   (idx[j] = original index of norm[j])
  encode_doc(text, tk)              -> token ids + each token's ORIGINAL char span (core, no
                                       leading space)
  bio_tags(spans, entities)         -> one BIO tag id per token (O, B-X, I-X; 25 tags)
  tags_to_entities(tags, spans)     -> [start, end, label] in original offsets
  make_windows(n, win, stride)      -> overlapping windows over long documents
  strict_f1(pred, gold)             -> micro P/R/F1 + per-label F1 (exact span + label)
"""
from __future__ import annotations

import hashlib
import json
import unicodedata
from collections import Counter, defaultdict

from common import PUNCT_MAP

LABELS = ['APP', 'RESP', 'A.COUNSEL', 'R.COUNSEL', 'JUDGE', 'WIT', 'AUTH', 'COURT',
          'STAT', 'PREC', 'DATE', 'CASENO']
TAGS = ["O"] + [f"{p}-{l}" for l in LABELS for p in ("B", "I")]     # 25 BIO tags
TAG2ID = {t: i for i, t in enumerate(TAGS)}


# ----------------------------------------------------------------------------- normalisation
def _norm_char(c: str) -> str:
    """Same rule as common.normalise, applied to ONE character (whitespace handled by caller)."""
    if c.isascii():
        return c
    s = unicodedata.normalize("NFKD", c.translate(PUNCT_MAP))
    return s.encode("ascii", "ignore").decode("ascii")


def normalise_with_map(text: str):
    """Return (norm, idx): norm == common.normalise(text) and idx[j] is the original index of
    norm[j]. A whitespace run becomes one space mapped to the run's first character."""
    out, idx, pending_space = [], [], None
    for i, ch in enumerate(text):
        if ch.isspace():
            if out and pending_space is None:
                pending_space = i                          # remember where the run started
            continue
        rep = _norm_char(ch)
        if not rep:
            continue                                      # dropped char (e.g. zero-width)
        if rep.isspace() or not rep.strip():
            if out and pending_space is None:
                pending_space = i
            continue
        if pending_space is not None:
            out.append(" "); idx.append(pending_space); pending_space = None
        for r in rep:
            if r.isspace():
                continue
            out.append(r); idx.append(i)
    return "".join(out), idx

Writing /kaggle/working/src/p2_ner.py


In [7]:
%%writefile -a /kaggle/working/src/p2_ner.py


# ----------------------------------------------------------------------------- tokens <-> chars
def encode_doc(text: str, tk):
    """Tokenise the normalised document. Returns (ids, spans) where spans[k] = (start, end) of
    token k's CORE (leading space stripped) in ORIGINAL offsets, or None for a pure-space token."""
    norm, idx = normalise_with_map(text)
    enc = tk.encode(norm, add_special_tokens=False)
    spans = []
    for s, e in enc.offsets:
        while s < e and norm[s] == " ":
            s += 1                                        # " court" -> core "court"
        spans.append((idx[s], idx[e - 1] + 1) if s < e else None)
    return enc.ids, spans


def bio_tags(spans, entities):
    """BIO tag id per token. A token is B-X if its core starts exactly at an entity start, I-X if
    it lies inside that entity, else O."""
    begin, inside = {}, {}
    for s, e, lab, *_ in entities:
        begin[s] = lab
        for c in range(s, e):
            inside[c] = (lab, s)
    tags = []
    for sp in spans:
        if sp is None:
            tags.append(0); continue
        s, _ = sp
        if s in begin:
            tags.append(TAG2ID[f"B-{begin[s]}"])
        elif s in inside:
            tags.append(TAG2ID[f"I-{inside[s][0]}"])
        else:
            tags.append(0)
    return tags


def tags_to_entities(tags, spans):
    """Decode BIO tags back to [start, end, label] (original offsets). An I-X that does not
    continue an X entity starts a new one (standard repair)."""
    ents, cur = [], None
    for t, sp in zip(tags, spans):
        if sp is None:
            continue                                      # spaces never break or extend a span
        tag = TAGS[t]
        if tag == "O":
            if cur: ents.append(cur); cur = None
            continue
        p, lab = tag.split("-", 1)
        if p == "B" or cur is None or cur[2] != lab:
            if cur: ents.append(cur)
            cur = [sp[0], sp[1], lab]
        else:
            cur[1] = sp[1]                                # extend the current entity
    if cur:
        ents.append(cur)
    return ents


def make_windows(n_tokens: int, win: int = 512, stride: int = 256):
    """Start offsets of overlapping windows covering [0, n_tokens)."""
    if n_tokens <= win:
        return [0]
    starts = list(range(0, n_tokens - win, stride))
    return starts + [n_tokens - win]


# ----------------------------------------------------------------------------- data + metric
def load_fold(text_path, gold_path=None, fold=""):
    T = json.load(open(text_path))
    G = json.load(open(gold_path)) if gold_path else {}
    return [{"id": str(k), "text": v, "ents": G.get(k, []), "fold": fold} for k, v in T.items()]


def dev_split(docs, n_dev_per_fold=5):
    """Deterministic held-out set: the n docs per fold with the smallest id-hash."""
    h = lambda d: hashlib.md5(d["id"].encode()).hexdigest()
    dev = set()
    for f in sorted({d["fold"] for d in docs}):
        dev |= {d["id"] for d in sorted((d for d in docs if d["fold"] == f), key=h)[:n_dev_per_fold]}
    return [d for d in docs if d["id"] not in dev], [d for d in docs if d["id"] in dev]


def strict_f1(pred: dict, gold: dict):
    """pred/gold: {doc_id: [[s, e, label, ...], ...]}. Exact span AND label must match."""
    tp, fp, fn = Counter(), Counter(), Counter()
    for d in set(gold) | set(pred):
        G = {(e[0], e[1], e[2]) for e in gold.get(d, [])}
        P = {(e[0], e[1], e[2]) for e in pred.get(d, [])}
        for x in P & G: tp[x[2]] += 1
        for x in P - G: fp[x[2]] += 1
        for x in G - P: fn[x[2]] += 1
    def f(a, b, c):
        p = a / max(1, a + b); r = a / max(1, a + c)
        return p, r, 2 * p * r / max(1e-9, p + r)
    P_, R_, F_ = f(sum(tp.values()), sum(fp.values()), sum(fn.values()))
    per = {l: round(f(tp[l], fp[l], fn[l])[2], 4) for l in LABELS}
    return {"precision": P_, "recall": R_, "f1": F_, "per_label_f1": per}

Appending to /kaggle/working/src/p2_ner.py


In [9]:
%%writefile -a /kaggle/working/src/p2_ner.py


# ----------------------------------------------------------------------------- v2: split glued punctuation
def encode_doc(text: str, tk, split_punct: bool = True):
    """Tokenise the normalised document. Returns (ids, spans) where spans[k] = (start, end) of
    token k's CORE (leading space stripped) in ORIGINAL offsets, or None for a pure-space token.
    split_punct: our BPE glues punctuation runs into one token (e.g. '.[' or '].'), which can
    straddle an entity boundary ('... & Ors.' + '[1]'). Such tokens are re-encoded one character
    at a time so every punctuation mark gets its own token and boundary."""
    norm, idx = normalise_with_map(text)
    enc = tk.encode(norm, add_special_tokens=False)
    ids, spans = [], []
    for tid, (s, e) in zip(enc.ids, enc.offsets):
        c = s
        while c < e and norm[c] == " ":
            c += 1                                        # " court" -> core "court"
        core = norm[c:e]
        if split_punct and len(core) > 1 and not any(ch.isalnum() for ch in core):
            for j, ch in enumerate(core):                 # one token per punctuation mark
                piece = (" " if j == 0 and c > s else "") + ch
                for sub in tk.encode(piece, add_special_tokens=False).ids:
                    ids.append(sub)
                    is_space = tk.decode([sub]).strip() == ""
                    spans.append(None if is_space else (idx[c + j], idx[c + j] + 1))
            continue
        ids.append(tid)
        spans.append((idx[c], idx[e - 1] + 1) if c < e else None)
    return ids, spans

Appending to /kaggle/working/src/p2_ner.py


In [10]:
# ===== Cell N3: pipeline check on real data -- oracle F1, lost entities, windows, tag balance =====
import sys, importlib, collections
sys.path.insert(0, f"{WORK}/src")
import common, p2_ner as ner
importlib.reload(ner)

# ---- all labelled docs + deterministic dev split ------------------------------------------------
docs = ner.load_fold(f"{D}/Train/fold1-text.json", f"{D}/Train/fold1-gold.json", "f1") + \
       ner.load_fold(f"{D}/Train/fold2-text.json", f"{D}/Train/fold2-gold.json", "f2")
train_docs, dev_docs = ner.dev_split(docs, 5)
print(f"{len(train_docs)} train docs | {len(dev_docs)} dev docs")

# ---- 1) the index-mapped normalisation must equal Phase 1's normalise() exactly -------------------
assert all(ner.normalise_with_map(d["text"])[0] == common.normalise(d["text"]) for d in docs)
print("[1] normalise_with_map == common.normalise on all 70 docs")

# ---- 2) oracle round trip: gold -> BIO -> spans (upper bound of any token tagger) -----------------
pred, gold, tagc, n_win, lost = {}, {}, collections.Counter(), 0, []
for d in docs:
    ids, spans = ner.encode_doc(d["text"], tk)
    tags = ner.bio_tags(spans, d["ents"])
    pred[d["id"]] = ner.tags_to_entities(tags, spans)
    gold[d["id"]] = d["ents"]
    tagc.update(ner.TAGS[t].split("-")[0] for t in tags)
    if d in train_docs:
        n_win += len(ner.make_windows(len(ids), 512, 256))
    P = {tuple(e[:3]) for e in pred[d["id"]]}
    lost += [(d["text"], e) for e in d["ents"] if tuple(e[:3]) not in P]
r = ner.strict_f1(pred, gold)
print(f"[2] oracle strict F1 = {r['f1']:.4f} (P {r['precision']:.4f}, R {r['recall']:.4f}) | "
      f"lost entities {len(lost)}")
print("    per label:", r["per_label_f1"])

# ---- 3) a few lost entities in context (boundary falls inside a token) -------------------------
for txt, (s, e, lab, *_) in lost[:8]:
    print(f"    {lab:9s} {txt[max(0, s-15):s]!r} [[{txt[s:e]!r}]] {txt[e:e+15]!r}")

# ---- 4) training size + tag balance ----------------------------------------------------------------
tot = sum(tagc.values())
print(f"[4] train windows (512/256): {n_win} | tag share: " +
      ", ".join(f"{k} {v/tot:.1%}" for k, v in tagc.items()))

60 train docs | 10 dev docs
[1] normalise_with_map == common.normalise on all 70 docs
[2] oracle strict F1 = 0.9927 (P 0.9927, R 0.9927) | lost entities 61
    per label: {'APP': 0.9853, 'RESP': 0.9948, 'A.COUNSEL': 1.0, 'R.COUNSEL': 1.0, 'JUDGE': 1.0, 'WIT': 1.0, 'AUTH': 0.991, 'COURT': 0.9822, 'STAT': 0.9849, 'PREC': 0.9949, 'DATE': 0.9994, 'CASENO': 1.0}
    AUTH      'truck down the ' [['Railway Board']] 's instructions '
    AUTH      'lementation of ' [['Railway Board']] 's decision to e'
    STAT      'ns". After the ' [['Constitution']] 'al Amendment (4'
    AUTH      ' including the ' [['State Government']] 's and the Unive'
    AUTH      'e views of the ' [['State Government']] 's and the Unive'
    COURT     'ature) and the ' [['High Court']] 's in India exer'
    AUTH      'In the case of ' [['Parliament']] 'ary legislation'
    STAT      'iance with the ' [['Constitution']] 'al man- date if'
[4] train windows (512/256): 1513 | tag share: O 90.4%, B 1.8%, I 7.9%


## P2.3 Token tagger on top of every Phase 1 backbone

`p2_model.py` wraps any of our four pretrained models as a **BIO token tagger**:

| Backbone | Hidden states used | Attention |
|---|---|---|
| M1 dense, M4 copy head | `tok_emb → 12 blocks → norm` (LM / copy head unused) | causal, or **bidirectional** |
| M2 MLA + MoE | same path through the MLA + MoE blocks | causal, or **bidirectional** |
| M3 encoder–decoder | its **encoder** only (bidirectional by design) | bidirectional |

A dropout + linear layer maps each hidden state to the 25 BIO tags.

**Causal vs bidirectional decoders.** A causal decoder tags each token seeing only the words *to its
left*, but entity boundaries often depend on what comes *next* ("State of Haryana" vs "State of
Haryana's counsel"). With `bidirectional=True`, the causal mask is removed during fine-tuning, so the
pretrained decoder becomes an encoder (as in LLM2Vec, 2024). We test both.

**Windows and prediction.** Training uses 512-token windows with stride 256. At prediction time each
token takes its label from the window where it has the most context: the most central position for
bidirectional models, the latest position (most left context) for causal ones.

**Training.** AdamW (backbone lr 5e-5, head lr 1e-3), linear warm-up and decay, fp16 autocast,
gradient clipping 1.0. Strict dev F1 is measured after every epoch, and the best epoch is kept.

In [11]:
%%writefile /kaggle/working/src/p2_model.py
"""
p2_model.py -- Phase 2 token tagger on top of ANY Phase-1 backbone (M1 dense, M2 MLA+MoE,
M3 encoder-decoder, M4 copy head) + windowed training / prediction.

  build_tagger(ckpt, bidirectional)  -> TokenTagger: backbone hidden states -> Linear -> 25 BIO tags
      * M1/M2/M4 decoders: tok_emb -> blocks -> norm_f (the LM head is not used)
      * M3: its bidirectional ENCODER only
      * bidirectional=True switches the decoders' attention from causal to full (every token sees
        its right context too, which matters for entity boundaries; cf. LLM2Vec, 2024)
  make_examples(docs, tk)            -> (ids, tags) windows for training
  predict_docs(model, docs, tk)      -> {doc_id: [[start, end, label], ...]} in original offsets
  train_tagger(...)                  -> fine-tunes, keeps the best dev-F1 weights
"""
from __future__ import annotations

import contextlib
import math
import random
import time

import torch
import torch.nn as nn
import torch.nn.functional as F

import p2_ner as ner

_SDPA = F.scaled_dot_product_attention


@contextlib.contextmanager
def full_attention():
    """Inside this block every scaled_dot_product_attention call ignores is_causal, so causal
    decoders attend in both directions. All our attention modules call F.scaled_dot_product_attention
    at run time, so swapping the function on the module is enough."""
    def sdpa(q, k, v, attn_mask=None, dropout_p=0.0, is_causal=False, **kw):
        return _SDPA(q, k, v, attn_mask=attn_mask, dropout_p=dropout_p, is_causal=False, **kw)
    F.scaled_dot_product_attention = sdpa
    try:
        yield
    finally:
        F.scaled_dot_product_attention = _SDPA


class TokenTagger(nn.Module):
    def __init__(self, backbone, arch: str, n_tags: int = len(ner.TAGS), dropout: float = 0.1,
                 bidirectional: bool = False):
        super().__init__()
        self.backbone, self.arch, self.bidirectional = backbone, arch, bidirectional
        d = backbone.cfg.d_model
        self.drop = nn.Dropout(dropout)
        self.head = nn.Linear(d, n_tags)
        nn.init.normal_(self.head.weight, 0.0, 0.02); nn.init.zeros_(self.head.bias)

    def hidden(self, ids):
        bb = self.backbone
        if self.arch == "encdec":
            return bb.encode(ids)                                   # M3: bidirectional encoder
        x = bb.tok_emb(ids)                                         # M1 / M2 / M4 decoders
        for blk in bb.blocks:
            x = blk(x, bb.rope_cos, bb.rope_sin)
        return bb.norm_f(x)

    def forward(self, ids):
        ctx = full_attention() if (self.bidirectional and self.arch != "encdec") else contextlib.nullcontext()
        with ctx:
            h = self.hidden(ids)
        return self.head(self.drop(h))                               # (B, T, n_tags)


def build_tagger(ckpt_path: str, bidirectional: bool = False, dropout: float = 0.1):
    """Load a Phase-1 best.pt and wrap it as a tagger (backbone weights in fp32)."""
    from train import build_model
    ck = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    bb = build_model(ck["arch"], ck["model_cfg"])
    bb.load_state_dict({k: v.float() for k, v in ck["model"].items()})
    if ck["arch"] == "encdec":
        del bb.dec                                                   # decoder is not needed for tagging
    return TokenTagger(bb, ck["arch"], dropout=dropout, bidirectional=bidirectional)


# ----------------------------------------------------------------------------- data
def encode_all(docs, tk):
    """Tokenise every document once: {id: (ids, spans, tags)}."""
    out = {}
    for d in docs:
        ids, spans = ner.encode_doc(d["text"], tk)
        out[d["id"]] = (ids, spans, ner.bio_tags(spans, d["ents"]))
    return out


def make_examples(enc, doc_ids, win=512, stride=256):
    ex = []
    for i in doc_ids:
        ids, _, tags = enc[i]
        for s in ner.make_windows(len(ids), win, stride):
            ex.append((ids[s:s + win], tags[s:s + win]))
    return ex


def _batch(examples, device):
    L = max(len(x) for x, _ in examples)
    ids = torch.zeros(len(examples), L, dtype=torch.long)
    tags = torch.full((len(examples), L), -100, dtype=torch.long)     # -100 = padding, ignored
    for b, (x, t) in enumerate(examples):
        ids[b, :len(x)] = torch.tensor(x); tags[b, :len(t)] = torch.tensor(t)
    return ids.to(device), tags.to(device)

Writing /kaggle/working/src/p2_model.py


In [12]:
%%writefile -a /kaggle/working/src/p2_model.py


# ----------------------------------------------------------------------------- prediction
@torch.no_grad()
def predict_logits(model, ids, win=512, stride=256, device="cuda", bs=16):
    """Per-token log-probs for one document. Each token takes its prediction from the window in
    which it has the most context: most central for bidirectional models, latest position (most
    left context) for causal decoders."""
    model.eval()
    causal = model.arch != "encdec" and not model.bidirectional
    starts = ner.make_windows(len(ids), win, stride)
    out = torch.zeros(len(ids), model.head.out_features)
    best = torch.full((len(ids),), -1e9)
    for b in range(0, len(starts), bs):
        chunk = starts[b:b + bs]
        x, _ = _batch([(ids[s:s + win], ids[s:s + win]) for s in chunk], device)
        with torch.autocast("cuda", dtype=torch.float16):
            lp = model(x).float().log_softmax(-1).cpu()
        for r, s in enumerate(chunk):
            n = min(win, len(ids) - s)
            j = torch.arange(n, dtype=torch.float)
            score = j if causal else torch.minimum(j, n - 1 - j)   # how much context position j has
            take = score > best[s:s + n]
            out[s:s + n][take] = lp[r, :n][take]
            best[s:s + n] = torch.maximum(best[s:s + n], score)
    return out


def predict_docs(model, docs, enc, win=512, stride=256, device="cuda"):
    preds = {}
    for d in docs:
        ids, spans, _ = enc[d["id"]]
        tags = predict_logits(model, ids, win, stride, device).argmax(-1).tolist()
        preds[d["id"]] = ner.tags_to_entities(tags, spans)
    return preds


# ----------------------------------------------------------------------------- training
def train_tagger(model, train_ex, dev_docs, enc, epochs=5, lr=5e-5, head_lr=1e-3, bs=8,
                 weight_decay=0.01, warmup=0.06, class_weight=None, device="cuda", seed=0, log=print):
    """AdamW, linear warm-up + linear decay, fp16 autocast. Evaluates strict F1 on dev_docs after
    every epoch and restores the best epoch's trainable weights at the end."""
    torch.manual_seed(seed); random.seed(seed)
    model.to(device)
    params = [p for p in model.parameters() if p.requires_grad]
    head_ids = {id(p) for p in model.head.parameters()}
    groups = [{"params": [p for p in params if id(p) not in head_ids], "lr": lr},
              {"params": [p for p in params if id(p) in head_ids], "lr": head_lr}]
    opt = torch.optim.AdamW(groups, weight_decay=weight_decay)
    steps = epochs * math.ceil(len(train_ex) / bs); warm = max(1, int(warmup * steps))
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: min((s + 1) / warm, max(0.0, (steps - s) / max(1, steps - warm))))
    scaler = torch.amp.GradScaler("cuda")
    w = None if class_weight is None else class_weight.to(device)
    best_f1, best_state, hist, step = -1.0, None, [], 0
    n_train = sum(p.numel() for p in params)
    log(f"trainable params: {n_train/1e6:.2f}M | {len(train_ex)} windows | {steps} steps")
    for ep in range(epochs):
        model.train(); random.shuffle(train_ex); t0 = time.time(); tot = 0.0
        for b in range(0, len(train_ex), bs):
            ids, tags = _batch(train_ex[b:b + bs], device)
            with torch.autocast("cuda", dtype=torch.float16):
                logits = model(ids)
            loss = F.cross_entropy(logits.float().view(-1, logits.size(-1)), tags.view(-1),
                                   weight=w, ignore_index=-100)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(params, 1.0)
            scaler.step(opt); scaler.update(); sched.step(); step += 1
            tot += loss.item()
        r = ner.strict_f1(predict_docs(model, dev_docs, enc, device=device),
                          {d["id"]: d["ents"] for d in dev_docs})
        hist.append({"epoch": ep + 1, "loss": tot / math.ceil(len(train_ex) / bs), **r})
        log(f"epoch {ep+1}: loss {hist[-1]['loss']:.4f} | dev strict F1 {r['f1']:.4f} "
            f"(P {r['precision']:.3f} R {r['recall']:.3f}) | {time.time()-t0:.0f}s")
        if r["f1"] > best_f1:
            best_f1 = r["f1"]
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best_state)
    return hist, best_f1

Appending to /kaggle/working/src/p2_model.py


## P2.4 First experiment: does the decoder need to see the right context?

M1 with **full fine-tuning**, trained twice with identical settings (5 epochs, 512/256 windows):
**causal** attention (as pretrained) vs **bidirectional** attention (causal mask removed).
This tells us which mode every later experiment (LoRA, LoRMA, adapters, KD) should use.

In [1]:
# ===== Cell N5: M1 full fine-tuning -- causal vs bidirectional attention =====
import sys, os, json, gc, importlib, torch
WORK = "/kaggle/working"; D = f"{WORK}/data_p2/Phase_2_LNER"
sys.path.insert(0, f"{WORK}/src")
import p2_ner as ner, p2_model as pm
importlib.reload(ner); importlib.reload(pm)
from tokenizers import Tokenizer

# ---- data: tokenise once, fixed dev split, training windows ----------------------------------------
tk = Tokenizer.from_file(f"{WORK}/tokenizer/hf_a1_bpe_v2_32k.json")
docs = ner.load_fold(f"{D}/Train/fold1-text.json", f"{D}/Train/fold1-gold.json", "f1") + \
       ner.load_fold(f"{D}/Train/fold2-text.json", f"{D}/Train/fold2-gold.json", "f2")
train_docs, dev_docs = ner.dev_split(docs, 5)
enc = pm.encode_all(docs, tk)
train_ex = pm.make_examples(enc, [d["id"] for d in train_docs])
print(f"{len(train_docs)} train docs -> {len(train_ex)} windows | {len(dev_docs)} dev docs")

# ---- results file shared by all Phase 2 experiments --------------------------------------------
RES = f"{WORK}/stats/p2_results.json"
results = json.load(open(RES)) if os.path.exists(RES) else {}
os.makedirs(f"{WORK}/p2_ckpt", exist_ok=True)

for bidir in (False, True):
    name = f"m1_full_{'bidir' if bidir else 'causal'}"
    print(f"\n===== {name} =====")
    model = pm.build_tagger(f"{WORK}/checkpoints/m1_dense_100m/best.pt", bidirectional=bidir)
    hist, f1 = pm.train_tagger(model, list(train_ex), dev_docs, enc, epochs=5, lr=5e-5)
    results[name] = {"best_dev_f1": f1, "hist": hist}
    json.dump(results, open(RES, "w"), indent=1)                    # save after every run
    torch.save({k: v.half() for k, v in model.state_dict().items()}, f"{WORK}/p2_ckpt/{name}.pt")
    del model; gc.collect(); torch.cuda.empty_cache()

print("\nbest dev strict F1:", {k: round(v["best_dev_f1"], 4) for k, v in results.items()})

60 train docs -> 1513 windows | 10 dev docs

===== m1_full_causal =====
trainable params: 100.11M | 1513 windows | 950 steps


/kaggle/working/src/p2_model.py:175: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scaler.step(opt); scaler.update(); sched.step(); step += 1


epoch 1: loss 0.3876 | dev strict F1 0.4022 (P 0.369 R 0.441) | 55s
epoch 2: loss 0.0323 | dev strict F1 0.4033 (P 0.372 R 0.441) | 57s
epoch 3: loss 0.0082 | dev strict F1 0.4507 (P 0.412 R 0.497) | 57s
epoch 4: loss 0.0030 | dev strict F1 0.4350 (P 0.401 R 0.475) | 57s
epoch 5: loss 0.0015 | dev strict F1 0.4506 (P 0.416 R 0.492) | 57s

===== m1_full_bidir =====
trainable params: 100.11M | 1513 windows | 950 steps
epoch 1: loss 0.3395 | dev strict F1 0.4152 (P 0.390 R 0.444) | 60s
epoch 2: loss 0.0353 | dev strict F1 0.4444 (P 0.421 R 0.471) | 59s
epoch 3: loss 0.0107 | dev strict F1 0.4508 (P 0.433 R 0.470) | 59s
epoch 4: loss 0.0037 | dev strict F1 0.4706 (P 0.449 R 0.494) | 59s
epoch 5: loss 0.0013 | dev strict F1 0.4797 (P 0.462 R 0.498) | 59s

best dev strict F1: {'m1_full_causal': 0.4507, 'm1_full_bidir': 0.4797}


## P2.4b Error analysis of the first model

Strict F1 hides *why* predictions fail. We sort every dev error into:
- **wrong label, exact span**: the right text, but a confused type (e.g. APP vs RESP);
- **boundary, same label**: the right type, but start/end off (fatal under strict scoring);
- **boundary + label** / **spurious** (no overlap with any gold entity) / **missed entirely**.

We also report **lenient F1** (any overlap and the same label counts). The gap between strict and
lenient F1 measures how much is lost purely to boundaries.

In [6]:
# ===== Cell N5b: error analysis of the best first run (m1_full_bidir) on the 10 dev docs =====
import collections, torch

# ---- reload the saved fine-tuned tagger and predict the dev docs ------------------------------
model = pm.build_tagger(f"{WORK}/checkpoints/m1_dense_100m/best.pt", bidirectional=True)
model.load_state_dict({k: v.float() for k, v in torch.load(f"{WORK}/p2_ckpt/m1_full_bidir.pt").items()})
model.cuda()
pred = pm.predict_docs(model, dev_docs, enc)
gold = {d["id"]: d["ents"] for d in dev_docs}
text = {d["id"]: d["text"] for d in dev_docs}
r = ner.strict_f1(pred, gold)
print(f"strict F1 {r['f1']:.4f} (P {r['precision']:.3f} R {r['recall']:.3f})")
print("per-label strict F1:", r["per_label_f1"])

# ---- classify every false positive / missed gold entity --------------------------------------
cat, conf, examples = collections.Counter(), collections.Counter(), collections.defaultdict(list)
len_tp = len_pred = len_gold = 0
for d in gold:
    G = [tuple(e[:3]) for e in gold[d]]; P = [tuple(e[:3]) for e in pred.get(d, [])]
    Gs, Ps = set(G), set(P)
    ov = lambda a, b: a[0] < b[1] and b[0] < a[1]                     # character overlap
    for p in Ps - Gs:
        hits = [g for g in G if ov(g, p)]
        if not hits:
            k = "FP spurious"
        elif any(g[:2] == p[:2] for g in hits):
            k = "FP wrong label (exact span)"
            conf[(next(g for g in hits if g[:2] == p[:2])[2], p[2])] += 1
        elif any(g[2] == p[2] for g in hits):
            k = "FP boundary (same label)"
            g = next(g for g in hits if g[2] == p[2])
            examples[k].append((p[2], text[d][g[0]:g[1]], text[d][p[0]:p[1]]))
        else:
            k = "FP boundary + label"
        cat[k] += 1
    cat["FN missed entirely"] += sum(not any(ov(g, p) for p in P) for g in Gs - Ps)
    # lenient: a gold entity counts if some prediction overlaps it with the same label
    len_tp += sum(any(ov(g, p) and g[2] == p[2] for p in P) for g in Gs)
    len_pred += len(Ps); len_gold += len(Gs)
lp, lr_ = len_tp / max(1, len_pred), len_tp / max(1, len_gold)
print(f"lenient F1 (overlap + same label): {2*lp*lr_/max(1e-9, lp+lr_):.4f}")
print("error types:", dict(cat.most_common()))
print("top label confusions (gold -> predicted):", conf.most_common(8))
print("boundary examples (label | gold | predicted):")
for lab, g, p in examples["FP boundary (same label)"][:10]:
    print(f"   {lab:9s} | {g[:60]!r} | {p[:60]!r}")
del model; torch.cuda.empty_cache()

strict F1 0.4803 (P 0.463 R 0.498)
per-label strict F1: {'APP': 0.087, 'RESP': 0.0583, 'A.COUNSEL': 0.1159, 'R.COUNSEL': 0.0851, 'JUDGE': 0.4167, 'WIT': 0.0, 'AUTH': 0.1717, 'COURT': 0.7064, 'STAT': 0.639, 'PREC': 0.263, 'DATE': 0.9474, 'CASENO': 0.2447}
lenient F1 (overlap + same label): 0.7179
error types: {'FP boundary (same label)': 402, 'FN missed entirely': 244, 'FP spurious': 215, 'FP boundary + label': 121, 'FP wrong label (exact span)': 30}
top label confusions (gold -> predicted): [(('RESP', 'AUTH'), 17), (('PREC', 'WIT'), 3), (('A.COUNSEL', 'R.COUNSEL'), 2), (('RESP', 'PREC'), 2), (('APP', 'AUTH'), 2), (('CASENO', 'PREC'), 1), (('JUDGE', 'PREC'), 1), (('PREC', 'R.COUNSEL'), 1)]
boundary examples (label | gold | predicted):
   STAT      | 'BT&AL Act' | '&AL'
   STAT      | 'BT&AL Act' | '&AL'
   COURT     | 'Civil Court' | 'Civil'
   COURT     | 'Civil Court' | 'Civil'
   COURT     | 'Court of Joint Civil Judge, Baroda' | 'Court'
   STAT      | 'T.P. Act' | '.'
   STAT      |

## P2.4c Constrained Viterbi decoding

The error analysis showed fragmented predictions (`BT&AL Act` → `&AL`), caused by choosing each token's tag
independently (argmax): sequences like `O, I-STAT, I-STAT` are allowed and get split into fragments.

**Constrained Viterbi decoding** finds the single most probable *valid* BIO sequence for the whole document,
where `I-X` may follow only `B-X` or `I-X` and a document cannot start with `I-X`. It's the same
inference a CRF uses, but applied to our existing per-token probabilities: **no retraining**.
Pure-space tokens are skipped (and ignored in the training loss), since they carry no characters.

In [7]:
%%writefile -a /kaggle/working/src/p2_ner.py


# ----------------------------------------------------------------------------- v3: constrained Viterbi decoding
def _allowed():
    """allowed[i, j]: tag j may follow tag i (BIO rule: I-X only after B-X or I-X)."""
    import numpy as np
    K = len(TAGS); A = np.ones((K, K), dtype=bool)
    for j, t in enumerate(TAGS):
        if t.startswith("I-"):
            lab = t[2:]
            A[:, j] = False
            A[TAG2ID[f"B-{lab}"], j] = A[TAG2ID[f"I-{lab}"], j] = True
    return A


def viterbi_bio(logp, spans):
    """Most probable VALID BIO sequence given per-token log-probs (T, K). Pure-space tokens
    (span None) are skipped, so they can neither break nor start an entity."""
    import numpy as np
    keep = [i for i, sp in enumerate(spans) if sp is not None]
    out = [0] * len(spans)
    if not keep:
        return out
    lp = np.asarray(logp)[keep]                                   # (n, K)
    A = _allowed(); NEG = -1e9
    trans = np.where(A, 0.0, NEG)                                 # (K, K)
    start = np.array([NEG if t.startswith("I-") else 0.0 for t in TAGS])
    n, K = lp.shape
    score = start + lp[0]; back = np.zeros((n, K), dtype=np.int32)
    for t in range(1, n):
        cand = score[:, None] + trans                             # prev x cur
        back[t] = cand.argmax(0)
        score = cand.max(0) + lp[t]
    best = [int(score.argmax())]
    for t in range(n - 1, 0, -1):
        best.append(int(back[t, best[-1]]))
    for i, tag in zip(keep, reversed(best)):
        out[i] = tag
    return out

Appending to /kaggle/working/src/p2_ner.py


In [8]:
%%writefile -a /kaggle/working/src/p2_model.py


# ----------------------------------------------------------------------------- v2: Viterbi decoding + ignore space tokens
def encode_all(docs, tk):
    """Tokenise every document once: {id: (ids, spans, tags)}. Pure-space tokens get tag -100
    (ignored in the loss): they carry no characters, so they have no entity label."""
    out = {}
    for d in docs:
        ids, spans = ner.encode_doc(d["text"], tk)
        tags = [t if sp is not None else -100 for t, sp in zip(ner.bio_tags(spans, d["ents"]), spans)]
        out[d["id"]] = (ids, spans, tags)
    return out


DECODE = "viterbi"                      # "argmax" (independent per token) or "viterbi" (valid BIO only)


def predict_docs(model, docs, enc, win=512, stride=256, device="cuda", decode=None):
    decode = decode or DECODE
    preds = {}
    for d in docs:
        ids, spans, _ = enc[d["id"]]
        lp = predict_logits(model, ids, win, stride, device)
        tags = ner.viterbi_bio(lp.numpy(), spans) if decode == "viterbi" else lp.argmax(-1).tolist()
        preds[d["id"]] = ner.tags_to_entities(tags, spans)
    return preds

Appending to /kaggle/working/src/p2_model.py


In [9]:
# ===== Cell N5d: same fine-tuned model, two decoders -- argmax vs constrained Viterbi =====
import importlib, torch
importlib.reload(ner); importlib.reload(pm)
enc = pm.encode_all(docs, tk)                                         # re-encode (space tokens -> ignored)
model = pm.build_tagger(f"{WORK}/checkpoints/m1_dense_100m/best.pt", bidirectional=True)
model.load_state_dict({k: v.float() for k, v in torch.load(f"{WORK}/p2_ckpt/m1_full_bidir.pt").items()})
model.cuda()
gold = {d["id"]: d["ents"] for d in dev_docs}
for dec in ("argmax", "viterbi"):
    r = ner.strict_f1(pm.predict_docs(model, dev_docs, enc, decode=dec), gold)
    print(f"{dec:8s} strict F1 {r['f1']:.4f} (P {r['precision']:.3f} R {r['recall']:.3f}) | "
          + " ".join(f"{k}:{v:.2f}" for k, v in r["per_label_f1"].items()))
del model; torch.cuda.empty_cache()

argmax   strict F1 0.4803 (P 0.463 R 0.498) | APP:0.09 RESP:0.06 A.COUNSEL:0.12 R.COUNSEL:0.09 JUDGE:0.42 WIT:0.00 AUTH:0.17 COURT:0.71 STAT:0.64 PREC:0.26 DATE:0.95 CASENO:0.24
viterbi  strict F1 0.5804 (P 0.626 R 0.541) | APP:0.14 RESP:0.11 A.COUNSEL:0.49 R.COUNSEL:0.42 JUDGE:0.58 WIT:0.00 AUTH:0.21 COURT:0.74 STAT:0.75 PREC:0.38 DATE:0.96 CASENO:0.31


## P2.4d Document header as context for every window

Who is the *appellant* and who the *respondent* is set in the judgment's **header**
("… Appellant(s) versus … Respondent(s)"), but a 512-token window deep in the document never sees it.
The model then can't tell a party (APP / RESP) from an authority or court (AUTH): per-label F1 was about 0.1 for APP/RESP.

**Fix:** every window after the first gets the **first 128 tokens of the document** + `<|sep|>` prepended:

`[ header (128) ] [SEP] [ window tokens (512) ]`

Header positions are excluded from the loss and from prediction. They only provide context, which a
bidirectional model can attend to from every position. We retrain M1 (full fine-tuning, bidirectional)
with this input and compare against the same model without the header (0.580 with Viterbi).

In [10]:
%%writefile -a /kaggle/working/src/p2_model.py


# ----------------------------------------------------------------------------- v3: document header as context
HEADER = 128                            # first HEADER tokens of the document are prepended to every window
SEP_ID = 3                              # <|sep|> (reserved special token from our A1-BPE tokenizer)


def _with_header(ids, s, win, header):
    """Window [s, s+win) with the document header in front (only when the window does not
    already start at the beginning). Returns (input ids, offset of the window inside them)."""
    if header and s > 0:
        return ids[:header] + [SEP_ID] + ids[s:s + win], header + 1
    return ids[s:s + win], 0


def make_examples(enc, doc_ids, win=512, stride=256, header=None):
    header = HEADER if header is None else header
    ex = []
    for i in doc_ids:
        ids, _, tags = enc[i]
        for s in ner.make_windows(len(ids), win, stride):
            x, off = _with_header(ids, s, win, header)
            ex.append((x, [-100] * off + tags[s:s + win]))        # header tokens are not trained on
    return ex


@torch.no_grad()
def predict_logits(model, ids, win=512, stride=256, device="cuda", bs=16, header=None):
    header = HEADER if header is None else header
    model.eval()
    causal = model.arch != "encdec" and not model.bidirectional
    starts = ner.make_windows(len(ids), win, stride)
    out = torch.zeros(len(ids), model.head.out_features)
    best = torch.full((len(ids),), -1e9)
    for b in range(0, len(starts), bs):
        chunk = starts[b:b + bs]
        packed = [_with_header(ids, s, win, header) for s in chunk]
        x, _ = _batch([(p, p) for p, _ in packed], device)
        with torch.autocast("cuda", dtype=torch.float16):
            lp = model(x).float().log_softmax(-1).cpu()
        for r, (s, (_, off)) in enumerate(zip(chunk, packed)):
            n = min(win, len(ids) - s)
            j = torch.arange(n, dtype=torch.float)
            score = j if causal else torch.minimum(j, n - 1 - j)
            take = score > best[s:s + n]
            out[s:s + n][take] = lp[r, off:off + n][take]
            best[s:s + n] = torch.maximum(best[s:s + n], score)
    return out

Appending to /kaggle/working/src/p2_model.py


In [11]:
# ===== Cell N8b: (1) is 128 header tokens enough? (2) M1 full fine-tuning, bidirectional + header =====
import importlib, statistics, gc, json, torch
importlib.reload(ner); importlib.reload(pm)
enc = pm.encode_all(docs, tk)

# ---- (1) token position where each document's FIRST appellant / respondent mention starts -------
first = []
for d in docs:
    ids, spans, _ = enc[d["id"]]
    starts = [e[0] for e in d["ents"] if e[2] in ("APP", "RESP")]
    if starts:
        c = min(starts)
        first.append(next(k for k, sp in enumerate(spans) if sp and sp[0] >= c))
first.sort()
print(f"first APP/RESP mention at token: median {statistics.median(first)} | "
      f"within 128: {sum(f < 128 for f in first)}/{len(first)} docs | max {first[-1]}")

# ---- (2) retrain with the header prepended to every window --------------------------------------
train_ex = pm.make_examples(enc, [d["id"] for d in train_docs])        # now with header context
name = "m1_full_bidir_header"
model = pm.build_tagger(f"{WORK}/checkpoints/m1_dense_100m/best.pt", bidirectional=True)
hist, f1 = pm.train_tagger(model, list(train_ex), dev_docs, enc, epochs=5, lr=5e-5)
r = ner.strict_f1(pm.predict_docs(model, dev_docs, enc), {d["id"]: d["ents"] for d in dev_docs})
print(f"\n{name}: strict F1 {r['f1']:.4f} (P {r['precision']:.3f} R {r['recall']:.3f}) | "
      + " ".join(f"{k}:{v:.2f}" for k, v in r["per_label_f1"].items()))

# ---- save results + weights ---------------------------------------------------------------------
results = json.load(open(RES))
results[name] = {"best_dev_f1": f1, "hist": hist, "final": r}
json.dump(results, open(RES, "w"), indent=1)
torch.save({k: v.half() for k, v in model.state_dict().items()}, f"{WORK}/p2_ckpt/{name}.pt")
del model; gc.collect(); torch.cuda.empty_cache()

first APP/RESP mention at token: median 32 | within 128: 65/67 docs | max 172
trainable params: 100.11M | 1513 windows | 950 steps
epoch 1: loss 0.3657 | dev strict F1 0.5358 (P 0.601 R 0.483) | 77s
epoch 2: loss 0.0360 | dev strict F1 0.5893 (P 0.621 R 0.561) | 76s
epoch 3: loss 0.0111 | dev strict F1 0.5755 (P 0.627 R 0.532) | 76s
epoch 4: loss 0.0033 | dev strict F1 0.5665 (P 0.612 R 0.527) | 76s
epoch 5: loss 0.0012 | dev strict F1 0.5742 (P 0.623 R 0.532) | 76s

m1_full_bidir_header: strict F1 0.5893 (P 0.621 R 0.561) | APP:0.04 RESP:0.06 A.COUNSEL:0.52 R.COUNSEL:0.38 JUDGE:0.04 WIT:0.00 AUTH:0.24 COURT:0.76 STAT:0.74 PREC:0.43 DATE:0.95 CASENO:0.48


In [13]:
# ===== Cell N8c: what happens on gold APP / RESP / JUDGE / WIT entities? (dev docs, saved header model) =====
import torch, collections
model = pm.build_tagger(f"{WORK}/checkpoints/m1_dense_100m/best.pt", bidirectional=True)
model.load_state_dict({k: v.float() for k, v in torch.load(f"{WORK}/p2_ckpt/m1_full_bidir_header.pt").items()})
model.cuda()
pred = pm.predict_docs(model, dev_docs, enc)
outcome = collections.Counter(); shown = collections.Counter()
for d in dev_docs:
    P = pred[d["id"]]; txt = d["text"]
    for s, e, lab, *_ in d["ents"]:
        if lab not in ("APP", "RESP", "JUDGE", "WIT"):
            continue
        hits = [p for p in P if p[0] < e and s < p[1]]                  # overlapping predictions
        res = ("exact" if [s, e, lab] in [p[:3] for p in hits] else
               "missed" if not hits else
               "label " + "/".join(sorted({p[2] for p in hits})))
        outcome[(lab, res)] += 1
        if res != "exact" and shown[lab] < 4:                             # a few examples per label
            shown[lab] += 1
            print(f"{lab:5s} gold {txt[s:e]!r:42s} -> {res:16s} pred {[txt[p[0]:p[1]] for p in hits][:2]} "
                  f"| ctx {txt[max(0, s-40):s].replace(chr(10), ' ')!r}")
print()
for lab in ("APP", "RESP", "JUDGE", "WIT"):
    print(lab, {r: n for (l, r), n in outcome.items() if l == lab})
del model; torch.cuda.empty_cache()

RESP  gold 'MAKWANA SHANABHAI BHIKHABHAI'             -> label RESP       pred ['MAKWANA SHANABHAI B'] | ctx "INH (DEAD) BY L.R'S. Vs. RESPONDENT: 1. "
RESP  gold 'MAKWANA PRABATBHAI'                       -> missed           pred [] | ctx 'T: 1. MAKWANA SHANABHAI BHIKHABHAI & 2. '
JUDGE gold 'VENKATACHALA N.'                          -> label JUDGE      pred ['VENKATACH'] | ctx 'TBHAI DATE OF JUDGMENT08/12/1994 BENCH: '
JUDGE gold 'VENKATACHALA N.'                          -> label JUDGE      pred ['VENKATACH'] | ctx '/1994 BENCH: VENKATACHALA N. (J) BENCH: '
JUDGE gold 'SAHAI, R.M.'                              -> missed           pred [] | ctx 'CHALA N. (J) BENCH: VENKATACHALA N. (J) '
APP   gold 'AIR LIQUIDE DEUTSCHLAND GmbH'             -> missed           pred [] | ctx '2014) MESSER GRIESHEIM GmbH (NOW CALLED '
JUDGE gold 'Rastogi'                                  -> missed           pred [] | ctx 'GASES PVT. LTD. …RESPONDENT(S) JUDGMENT '
APP   gold 'Sardar Gurmeet Singh'       

## P2.4e Word-consistent decoding

The error analysis showed entities **cut inside a word**: `VENKATACHALA N.` → `VENKATACH`,
`MAKWANA SHANABHAI BHIKHABHAI` → `MAKWANA SHANABHAI B`. Rare all-caps names are split into many BPE
pieces, and the model loses confidence on later pieces. A valid BIO sequence can then still end mid-word.

**Word-consistent Viterbi:** sub-word pieces that touch (no space, letters/digits on both sides) form one
word. The decoder chooses one tag per *word*, with the first piece B-X (or I-X) and the remaining pieces I-X,
or all O. A word's score for a tag is the exact joint log-probability of that consistent assignment, so an
entity can never start or end inside a word. Again, there is **no retraining**.

In [14]:
%%writefile -a /kaggle/working/src/p2_ner.py


# ----------------------------------------------------------------------------- v4: word-consistent Viterbi
def word_groups(spans, text):
    """Group sub-word tokens into words: token k continues the previous token's word if they touch
    (no space) and both sides are letters/digits (e.g. 'VENKAT' + 'ACH' + 'ALA')."""
    groups, prev = [], None
    for k, sp in enumerate(spans):
        if sp is None:
            continue
        if (prev is not None and sp[0] == prev[1] and text[prev[1] - 1].isalnum()
                and text[sp[0]].isalnum()):
            groups[-1].append(k)
        else:
            groups.append([k])
        prev = sp
    return groups


def _cont():
    """Tag that a word's continuation pieces must take: B-X/I-X -> I-X, O -> O."""
    return [0 if t == "O" else TAG2ID["I-" + t[2:]] for t in TAGS]


def viterbi_words(logp, spans, text):
    """Constrained Viterbi over WORDS: every sub-word piece of a word gets a consistent tag
    (first piece B-X or I-X, the rest I-X; or all O). A word's score for tag t is the exact joint
    log-prob of that consistent assignment, so a name can no longer be cut inside a word."""
    import numpy as np
    lp = np.asarray(logp)
    groups = word_groups(spans, text)
    out = [0] * len(spans)
    if not groups:
        return out
    cont = np.array(_cont())
    W = np.stack([lp[g[0]] + (lp[g[1:]][:, cont].sum(0) if len(g) > 1 else 0.0) for g in groups])
    A = _allowed(); NEG = -1e9
    trans = np.where(A, 0.0, NEG)
    start = np.array([NEG if t.startswith("I-") else 0.0 for t in TAGS])
    n = len(groups)
    score = start + W[0]; back = np.zeros((n, len(TAGS)), dtype=np.int32)
    for t in range(1, n):
        cand = score[:, None] + trans
        back[t] = cand.argmax(0); score = cand.max(0) + W[t]
    best = [int(score.argmax())]
    for t in range(n - 1, 0, -1):
        best.append(int(back[t, best[-1]]))
    for g, tag in zip(groups, reversed(best)):
        out[g[0]] = tag
        for k in g[1:]:
            out[k] = int(cont[tag])
    return out

Appending to /kaggle/working/src/p2_ner.py


In [15]:
%%writefile -a /kaggle/working/src/p2_model.py


# ----------------------------------------------------------------------------- v4: word-level decoding
DECODE = "words"                        # "argmax" | "viterbi" (token level) | "words" (word-consistent)


def predict_docs(model, docs, enc, win=512, stride=256, device="cuda", decode=None):
    decode = decode or DECODE
    preds = {}
    for d in docs:
        ids, spans, _ = enc[d["id"]]
        lp = predict_logits(model, ids, win, stride, device).numpy()
        if decode == "words":
            tags = ner.viterbi_words(lp, spans, d["text"])
        elif decode == "viterbi":
            tags = ner.viterbi_bio(lp, spans)
        else:
            tags = lp.argmax(-1).tolist()
        preds[d["id"]] = ner.tags_to_entities(tags, spans)
    return preds

Appending to /kaggle/working/src/p2_model.py


In [16]:
# ===== Cell N8e: argmax vs token Viterbi vs word Viterbi, with and without header context =====
import importlib, torch
importlib.reload(ner); importlib.reload(pm)
gold = {d["id"]: d["ents"] for d in dev_docs}
for name, header in [("m1_full_bidir", 0), ("m1_full_bidir_header", 128)]:
    pm.HEADER = header                                                  # predict exactly as trained
    model = pm.build_tagger(f"{WORK}/checkpoints/m1_dense_100m/best.pt", bidirectional=True)
    model.load_state_dict({k: v.float() for k, v in torch.load(f"{WORK}/p2_ckpt/{name}.pt").items()})
    model.cuda()
    for dec in ("argmax", "viterbi", "words"):
        r = ner.strict_f1(pm.predict_docs(model, dev_docs, enc, decode=dec), gold)
        print(f"{name:22s} {dec:8s} strict F1 {r['f1']:.4f} (P {r['precision']:.3f} R {r['recall']:.3f}) | "
              + " ".join(f"{k}:{v:.2f}" for k, v in r["per_label_f1"].items()))
    del model; torch.cuda.empty_cache()
pm.HEADER = 128

m1_full_bidir          argmax   strict F1 0.4803 (P 0.463 R 0.498) | APP:0.09 RESP:0.06 A.COUNSEL:0.12 R.COUNSEL:0.09 JUDGE:0.42 WIT:0.00 AUTH:0.17 COURT:0.71 STAT:0.64 PREC:0.26 DATE:0.95 CASENO:0.24
m1_full_bidir          viterbi  strict F1 0.5804 (P 0.626 R 0.541) | APP:0.14 RESP:0.11 A.COUNSEL:0.49 R.COUNSEL:0.42 JUDGE:0.58 WIT:0.00 AUTH:0.21 COURT:0.74 STAT:0.75 PREC:0.38 DATE:0.96 CASENO:0.31
m1_full_bidir          words    strict F1 0.5852 (P 0.636 R 0.542) | APP:0.23 RESP:0.11 A.COUNSEL:0.55 R.COUNSEL:0.44 JUDGE:0.62 WIT:0.00 AUTH:0.21 COURT:0.74 STAT:0.75 PREC:0.36 DATE:0.95 CASENO:0.31
m1_full_bidir_header   argmax   strict F1 0.4732 (P 0.438 R 0.515) | APP:0.00 RESP:0.05 A.COUNSEL:0.19 R.COUNSEL:0.04 JUDGE:0.03 WIT:0.00 AUTH:0.17 COURT:0.74 STAT:0.63 PREC:0.22 DATE:0.94 CASENO:0.37
m1_full_bidir_header   viterbi  strict F1 0.5893 (P 0.621 R 0.561) | APP:0.04 RESP:0.06 A.COUNSEL:0.52 R.COUNSEL:0.38 JUDGE:0.04 WIT:0.00 AUTH:0.24 COURT:0.76 STAT:0.74 PREC:0.43 DATE:0.95 CASENO:

In [12]:
import json; D = "/kaggle/working/data_p2/Phase_2_LNER"
print({f: len(json.load(open(f"{D}/{f}"))) for f in ["Train/fold1-text.json", "Train/fold2-text.json", "Test/fold3-text.json"]})

{'Train/fold1-text.json': 35, 'Train/fold2-text.json': 35, 'Test/fold3-text.json': 35}


## P2.5 Adaptation methods and knowledge distillation (`p2_peft.py`)

All methods freeze the pretrained backbone (except *full*) and train only their own parameters plus the tagging head.

| Method | What is trained | Reference |
|---|---|---|
| Full fine-tuning | everything | – |
| Partial (top-k layers) | last 4 transformer layers | – |
| Norm-gain (BitFit-style) | RMSNorm gains only (our models have no bias vectors) | Ben Zaken et al. 2022 |
| LoRA | low-rank $\Delta W = \tfrac{\alpha}{r}BA$ added to query + value | Hu et al. 2022 |
| **LoRMA+** | $h = (I + \tfrac{\alpha}{r}BA)\,W_0x$, $B = 0$ at init | Bihany et al. 2025 |
| **LoRMAπ** | $h = \mathcal{I}_\pi(BA)\,W_0x$, rows of $BA$ cyclically rotated; $\mathcal{I}_\pi(BA) = I$ at init | Bihany et al. 2025 |
| Adapters | bottleneck MLP after every layer (64 dims), up-projection = 0 at init | Houlsby 2019 / Pfeiffer 2020 |
| Prompt tuning | 20 soft-prompt vectors prepended to the input | Lester et al. 2021 |
| **Knowledge distillation** | 6-layer student (every other M1 layer) learns from a fine-tuned teacher | Hinton et al. 2015; DistilBERT |

LoRA and LoRMA use **r = 8, α = r** on the **query and value** projections, as in the LoRMA paper.
For M2 (MLA), "value" is the latent up-projection `w_uv`. **LoRA is additive, LoRMA is multiplicative**:
comparing them on the same backbones is one of the required analyses.

**Sanity check (Cell N7):** every method except prompt tuning must reproduce the pretrained hidden states
*exactly* at initialisation, and we report each method's trainable-parameter count.

In [4]:
%%writefile -a /kaggle/working/src/p2_peft.py


# ----------------------------------------------------------------------------- dispatcher
def apply_method(tagger, method, r=8, alpha=8, top_k=4, bottleneck=64, n_prompt=20):
    if method == "full":
        return tagger
    for p in tagger.backbone.parameters():
        p.requires_grad = False
    if method == "partial":
        for layer in list(_layers(tagger))[-top_k:]:
            for p in layer.parameters():
                p.requires_grad = True
    elif method == "normgain":
        for n_, p in tagger.backbone.named_parameters():
            if "norm" in n_:
                p.requires_grad = True
    elif method == "lora":
        _wrap_linears(tagger, lambda l: LoRALinear(l, r, alpha))
    elif method == "lorma_plus":
        _wrap_linears(tagger, lambda l: LoRMAPlusLinear(l, r, alpha))
    elif method == "lorma_pi":
        _wrap_linears(tagger, lambda l: LoRMAPiLinear(l, r))
    elif method == "adapter":
        L = _layers(tagger); d = tagger.backbone.cfg.d_model
        for i in range(len(L)):
            L[i] = AdapterLayer(L[i], d, bottleneck)
    elif method == "prompt":
        bb = tagger.backbone
        bb.tok_emb = PromptEmbedding(bb.tok_emb, n_prompt)
        tagger.head = DropPrefix(tagger.head, n_prompt)
    else:
        raise ValueError(method)
    for p in tagger.head.parameters():
        p.requires_grad = True
    return tagger


# ----------------------------------------------------------------------------- distillation
def shrink_dense(tagger, keep=(0, 2, 4, 6, 8, 10)):
    """Student from a decoder tagger: keep only the listed layers (every other layer, as in
    DistilBERT's initialisation)."""
    bb = tagger.backbone
    bb.blocks = nn.ModuleList([bb.blocks[i] for i in keep])
    return tagger


def train_kd(student, teacher, train_ex, dev_docs, enc, epochs=5, lr=1e-4, head_lr=1e-3, bs=8,
             T=2.0, lam=0.5, device="cuda", seed=0, log=print):
    """loss = (1-lam) * CE(student, gold) + lam * T^2 * KL(teacher_T || student_T), padding ignored."""
    torch.manual_seed(seed); random.seed(seed)
    student.to(device); teacher.to(device).eval()
    params = [p for p in student.parameters() if p.requires_grad]
    head_ids = {id(p) for p in student.head.parameters()}
    opt = torch.optim.AdamW([{"params": [p for p in params if id(p) not in head_ids], "lr": lr},
                             {"params": [p for p in params if id(p) in head_ids], "lr": head_lr}],
                            weight_decay=0.01)
    steps = epochs * math.ceil(len(train_ex) / bs); warm = max(1, int(0.06 * steps))
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: min((s + 1) / warm, max(0.0, (steps - s) / max(1, steps - warm))))
    scaler = torch.amp.GradScaler("cuda")
    best_f1, best_state, hist = -1.0, None, []
    log(f"student trainable {sum(p.numel() for p in params)/1e6:.2f}M | {len(train_ex)} windows")
    for ep in range(epochs):
        student.train(); random.shuffle(train_ex); t0 = time.time(); tot = 0.0
        for b in range(0, len(train_ex), bs):
            ids, tags = pm._batch(train_ex[b:b + bs], device)
            mask = tags != -100
            with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
                t_logits = teacher(ids).float()
            with torch.autocast("cuda", dtype=torch.float16):
                s_logits = student(ids).float()
            ce = F.cross_entropy(s_logits[mask], tags[mask])
            kl = F.kl_div(F.log_softmax(s_logits[mask] / T, -1), F.log_softmax(t_logits[mask] / T, -1),
                          log_target=True, reduction="batchmean") * T * T
            loss = (1 - lam) * ce + lam * kl
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward(); scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(params, 1.0)
            scaler.step(opt); scaler.update(); sched.step(); tot += loss.item()
        r = ner.strict_f1(pm.predict_docs(student, dev_docs, enc, device=device),
                          {d["id"]: d["ents"] for d in dev_docs})
        hist.append({"epoch": ep + 1, "loss": tot / math.ceil(len(train_ex) / bs), **r})
        log(f"epoch {ep+1}: loss {hist[-1]['loss']:.4f} | dev strict F1 {r['f1']:.4f} "
            f"(P {r['precision']:.3f} R {r['recall']:.3f}) | {time.time()-t0:.0f}s")
        if r["f1"] > best_f1:
            best_f1 = r["f1"]; best_state = {k: v.detach().cpu().clone() for k, v in student.state_dict().items()}
    student.load_state_dict(best_state)
    return hist, best_f1

Appending to /kaggle/working/src/p2_peft.py


In [5]:
# ===== Cell N7: adaptation methods -- identity at initialisation + trainable-parameter counts =====
import importlib, torch, gc
import p2_peft as pe
importlib.reload(pe)
CK = f"{WORK}/checkpoints/m1_dense_100m/best.pt"
x = torch.tensor([train_ex[0][0][:256]], device="cuda")            # one real 256-token window

# ---- reference: hidden states of the untouched pretrained backbone -----------------------------
ref = pm.build_tagger(CK).cuda().eval()
with torch.no_grad():
    h0 = ref.hidden(x).float()
del ref; torch.cuda.empty_cache()

# ---- every method must start exactly at the pretrained model (prompt tuning cannot: it adds inputs)
for m in ["full", "partial", "normgain", "lora", "lorma_plus", "lorma_pi", "adapter", "prompt"]:
    t = pe.apply_method(pm.build_tagger(CK), m).cuda().eval()
    tr, tot = pe.count(t)
    with torch.no_grad():
        h = t.hidden(x).float()[:, -h0.size(1):]                       # drop soft-prompt positions
    diff = (h - h0).abs().max().item()
    flag = "" if m == "prompt" else ("OK" if diff < 1e-4 else "<-- NOT identity!")
    print(f"{m:11s} trainable {tr/1e6:8.3f}M ({100*tr/tot:6.2f}%) | max|h - h_pretrained| {diff:.1e} {flag}")
    del t; gc.collect(); torch.cuda.empty_cache()

full        trainable  100.113M (100.00%) | max|h - h_pretrained| 0.0e+00 OK
partial     trainable   25.192M ( 25.16%) | max|h - h_pretrained| 0.0e+00 OK
normgain    trainable    0.040M (  0.04%) | max|h - h_pretrained| 0.0e+00 OK
lora        trainable    0.265M (  0.26%) | max|h - h_pretrained| 0.0e+00 OK
lorma_plus  trainable    0.216M (  0.22%) | max|h - h_pretrained| 0.0e+00 OK
lorma_pi    trainable    0.216M (  0.22%) | max|h - h_pretrained| 0.0e+00 OK
adapter     trainable    1.209M (  1.19%) | max|h - h_pretrained| 0.0e+00 OK
prompt      trainable    0.035M (  0.03%) | max|h - h_pretrained| 9.3e+00 


## P2.6 Main experiment batch: adaptation methods across all four backbones

**Fixed pipeline** (from P2.4): bidirectional attention, 512/256 windows, word-consistent Viterbi decoding,
same dev split (10 docs). The header context was tested and dropped (+0.003, within noise, +33% cost).

**Runs** (each a substantially different method or backbone, as the spec requires):

| Run | Backbone | Method | Why |
|---|---|---|---|
| baseline | M1 dense | full fine-tuning | reference (0.585) |
| 1 | M1 | LoRA (r = 8) | additive low-rank |
| 2 | M1 | **LoRMA+** (r = 8) | multiplicative, identity inflation (compulsory) |
| 3 | M1 | **LoRMAπ** (r = 8) | multiplicative, permutation inflation |
| 4 | M3 encoder | full fine-tuning | natively bidirectional encoder |
| 5 | M2 MLA + MoE | LoRMA+ | efficient backbone, multiplicative adaptation |
| 6 | M4 copy-head decoder | adapters | bottleneck adapters |
| 7 | KD student (6 layers) | distillation from the best teacher | compulsory KD (next cell) |

Runs 1–3 use the **same backbone and identical settings**, so they give a clean
**LoRA vs LoRMA+ vs LoRMAπ** comparison. Learning rates follow common practice for each method
(LoRA/LoRMA ~5e-4, following the LoRMA paper's 2e-4 to 5e-4; LoRMAπ lower, as in the paper).

In [17]:
# ===== Cell N9: Phase 2 experiment batch -- one run after another, everything saved after each =====
import importlib, gc, json, time, traceback, torch
import p2_peft as pe
importlib.reload(ner); importlib.reload(pm); importlib.reload(pe)

# ---- final pipeline settings -----------------------------------------------------------------
pm.HEADER = 0                                # header context dropped (no clear gain, +33% cost)
pm.DECODE = "words"                          # word-consistent Viterbi
enc = pm.encode_all(docs, tk)
train_ex = pm.make_examples(enc, [d["id"] for d in train_docs])
gold = {d["id"]: d["ents"] for d in dev_docs}

CK = {"m1": "m1_dense_100m", "m2": "m2_mla_moe", "m3": "m3_encdec", "m4": "m4_copy"}
# method -> (backbone learning rate, epochs). Small trainable sets need higher lr + more epochs.
LR = {"full": (5e-5, 5), "partial": (1e-4, 6), "normgain": (2e-3, 10), "lora": (5e-4, 10),
      "lorma_plus": (5e-4, 10), "lorma_pi": (2e-4, 10), "adapter": (5e-4, 10), "prompt": (3e-3, 10)}
PLAN = [("m1", "lora"), ("m1", "lorma_plus"), ("m1", "lorma_pi"),
        ("m3", "full"), ("m2", "lorma_plus"), ("m4", "adapter")]

def run(bb, method):
    """Train one (backbone, method) pair; skip it if it is already in the results file."""
    name = f"{bb}_{method}"
    if name in json.load(open(RES)):
        print("skip (already done):", name); return
    lr, ep = LR[method]
    t = pe.apply_method(pm.build_tagger(f"{WORK}/checkpoints/{CK[bb]}/best.pt", bidirectional=True), method)
    tr, tot = pe.count(t)
    print(f"\n===== {name} | trainable {tr/1e6:.3f}M of {tot/1e6:.1f}M | lr {lr} | {ep} epochs =====", flush=True)
    t0 = time.time()
    hist, f1 = pm.train_tagger(t, list(train_ex), dev_docs, enc, epochs=ep, lr=lr)
    r = ner.strict_f1(pm.predict_docs(t, dev_docs, enc), gold)        # best epoch, word Viterbi
    results = json.load(open(RES))
    results[name] = {"backbone": bb, "method": method, "trainable": tr, "total": tot, "lr": lr,
                     "epochs": ep, "minutes": round((time.time() - t0) / 60, 1),
                     "best_dev_f1": f1, "final": r, "hist": hist}
    json.dump(results, open(RES, "w"), indent=1)
    torch.save({k: v.half() for k, v in t.state_dict().items()}, f"{WORK}/p2_ckpt/{name}.pt")
    print(f">>> {name}: dev strict F1 {r['f1']:.4f}  ({results[name]['minutes']} min)", flush=True)
    del t; gc.collect(); torch.cuda.empty_cache()

for bb, m in PLAN:
    try:
        run(bb, m)
    except Exception:                                    # one failing run must not stop the batch
        traceback.print_exc(); gc.collect(); torch.cuda.empty_cache()


===== m1_lora | trainable 0.265M of 100.4M | lr 0.0005 | 10 epochs =====
trainable params: 0.26M | 1513 windows | 1900 steps
epoch 1: loss 0.7301 | dev strict F1 0.5309 (P 0.549 R 0.514) | 49s
epoch 2: loss 0.0996 | dev strict F1 0.5593 (P 0.592 R 0.530) | 48s
epoch 3: loss 0.0805 | dev strict F1 0.5773 (P 0.639 R 0.526) | 48s
epoch 4: loss 0.0673 | dev strict F1 0.6193 (P 0.648 R 0.593) | 48s
epoch 5: loss 0.0580 | dev strict F1 0.6027 (P 0.626 R 0.581) | 48s
epoch 6: loss 0.0505 | dev strict F1 0.6021 (P 0.637 R 0.571) | 48s
epoch 7: loss 0.0432 | dev strict F1 0.5647 (P 0.623 R 0.517) | 48s
epoch 8: loss 0.0390 | dev strict F1 0.5758 (P 0.619 R 0.538) | 48s
epoch 9: loss 0.0341 | dev strict F1 0.5978 (P 0.631 R 0.568) | 48s
epoch 10: loss 0.0314 | dev strict F1 0.5926 (P 0.627 R 0.562) | 48s
>>> m1_lora: dev strict F1 0.6193  (8.1 min)

===== m1_lorma_plus | trainable 0.216M of 100.3M | lr 0.0005 | 10 epochs =====
trainable params: 0.22M | 1513 windows | 1900 steps
epoch 1: loss 0.

In [ ]:
# ===== Cell N9b: all Phase 2 runs so far, best first =====
import json, pandas as pd
res = json.load(open(RES))
rows = {k: {"backbone": v.get("backbone", "m1"), "method": v.get("method", "full"),
            "trainable %": round(100 * v["trainable"] / v["total"], 2) if "trainable" in v else 100.0,
            "dev strict F1": round(v.get("final", {}).get("f1", v["best_dev_f1"]), 4),
            "P": round(v.get("final", {}).get("precision", float("nan")), 3),
            "R": round(v.get("final", {}).get("recall", float("nan")), 3),
            "min": v.get("minutes")} for k, v in res.items()}
display(pd.DataFrame(rows).T.sort_values("dev strict F1", ascending=False))

## P2.7 Knowledge distillation (compulsory)

**Teacher:** M1 + LoRA (100.4M parameters, best dev model). **Student:** M1 reduced to 6 of its 12 layers
(layers 0, 2, 4, 6, 8, 10, as in DistilBERT's initialisation), about 62M parameters, fully fine-tuned.

**Loss:** $\mathcal{L} = (1-\lambda)\,\mathrm{CE}(\text{student}, \text{gold}) + \lambda\,T^2\,
\mathrm{KL}\big(p_{\text{teacher}}^{T} \,\|\, p_{\text{student}}^{T}\big)$ with $\lambda = 0.5$, $T = 2$, on every
non-padding token. Soft targets carry the teacher's uncertainty between related labels (e.g. APP vs RESP vs AUTH).

**Control:** the identical student trained with $\lambda = 0$ (gold labels only). The difference isolates the
effect of distillation. We also report parameters and inference time (teacher vs student).

In [28]:
# ===== Cell N11: knowledge distillation -- teacher M1+LoRA (100M) -> 6-layer M1 student (~62M) =====
import importlib, gc, json, time, torch
import p2_peft as pe
importlib.reload(pe)
CK1 = f"{WORK}/checkpoints/m1_dense_100m/best.pt"

# ---- teacher: rebuild M1 + LoRA and load its fine-tuned weights -----------------------------
teacher = pe.apply_method(pm.build_tagger(CK1, bidirectional=True), "lora")
teacher.load_state_dict({k: v.float() for k, v in torch.load(f"{WORK}/p2_ckpt/m1_lora.pt").items()})
teacher.cuda().eval()
t0 = time.time(); r_t = ner.strict_f1(pm.predict_docs(teacher, dev_docs, enc), gold); t_teacher = time.time() - t0
print(f"teacher: dev F1 {r_t['f1']:.4f} | {pe.count(teacher)[1]/1e6:.1f}M params | dev inference {t_teacher:.1f}s")

# ---- student with KD (lam 0.5) and the same student without KD (lam 0) ------------------------
for name, lam in [("kd_student_m1x6", 0.5), ("nokd_student_m1x6", 0.0)]:
    if name in json.load(open(RES)):
        print("skip (already done):", name); continue
    student = pe.shrink_dense(pm.build_tagger(CK1, bidirectional=True))   # every other layer
    tr, tot = pe.count(student)
    print(f"\n===== {name} | {tot/1e6:.1f}M params (teacher {pe.count(teacher)[1]/1e6:.1f}M) | lambda {lam} =====")
    t0 = time.time()
    hist, f1 = pe.train_kd(student, teacher, list(train_ex), dev_docs, enc, epochs=8, lr=1e-4, T=2.0, lam=lam)
    minutes = round((time.time() - t0) / 60, 1)
    t1 = time.time(); r = ner.strict_f1(pm.predict_docs(student, dev_docs, enc), gold); t_student = time.time() - t1
    results = json.load(open(RES))
    results[name] = {"backbone": "m1x6", "method": "kd" if lam else "student_no_kd", "trainable": tr,
                     "total": tot, "lr": 1e-4, "epochs": 8, "minutes": minutes, "lambda": lam,
                     "best_dev_f1": f1, "final": r, "hist": hist,
                     "dev_inference_s": round(t_student, 1), "teacher_inference_s": round(t_teacher, 1)}
    json.dump(results, open(RES, "w"), indent=1)
    torch.save({k: v.half() for k, v in student.state_dict().items()}, f"{WORK}/p2_ckpt/{name}.pt")
    print(f">>> {name}: dev strict F1 {r['f1']:.4f} | inference {t_student:.1f}s vs teacher {t_teacher:.1f}s")
    del student; gc.collect(); torch.cuda.empty_cache()
del teacher; gc.collect(); torch.cuda.empty_cache()

teacher: dev F1 0.6193 | 100.4M params | dev inference 5.6s

===== kd_student_m1x6 | 62.4M params (teacher 100.4M) | lambda 0.5 =====
student trainable 62.35M | 1513 windows


/kaggle/working/src/p2_peft.py:212: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scaler.step(opt); scaler.update(); sched.step(); tot += loss.item()


epoch 1: loss 0.7495 | dev strict F1 0.5790 (P 0.646 R 0.525) | 49s
epoch 2: loss 0.0635 | dev strict F1 0.6146 (P 0.678 R 0.562) | 50s
epoch 3: loss 0.0456 | dev strict F1 0.6103 (P 0.664 R 0.565) | 50s
epoch 4: loss 0.0381 | dev strict F1 0.5807 (P 0.661 R 0.518) | 49s
epoch 5: loss 0.0339 | dev strict F1 0.5996 (P 0.674 R 0.540) | 51s
epoch 6: loss 0.0311 | dev strict F1 0.5859 (P 0.654 R 0.531) | 51s
epoch 7: loss 0.0296 | dev strict F1 0.5941 (P 0.664 R 0.538) | 51s
epoch 8: loss 0.0289 | dev strict F1 0.5970 (P 0.662 R 0.544) | 51s
>>> kd_student_m1x6: dev strict F1 0.6146 | inference 4.4s vs teacher 5.6s

===== nokd_student_m1x6 | 62.4M params (teacher 100.4M) | lambda 0.0 =====
student trainable 62.35M | 1513 windows
epoch 1: loss 0.3546 | dev strict F1 0.5515 (P 0.612 R 0.502) | 51s
epoch 2: loss 0.0362 | dev strict F1 0.5759 (P 0.653 R 0.515) | 50s
epoch 3: loss 0.0125 | dev strict F1 0.6015 (P 0.670 R 0.546) | 51s
epoch 4: loss 0.0036 | dev strict F1 0.6026 (P 0.678 R 0.542)

In [29]:
%%writefile -a /kaggle/working/src/p2_ner.py


# ----------------------------------------------------------------------------- v5: strict MACRO-F1 (official metric)
def strict_f1(pred: dict, gold: dict):
    """Exact span AND label. Returns micro P/R/F1, per-label F1, and MACRO-F1 (the official
    Phase-2 metric) over the labels that occur in the gold data (and over all 12, for reference)."""
    tp, fp, fn = Counter(), Counter(), Counter()
    for d in set(gold) | set(pred):
        G = {(e[0], e[1], e[2]) for e in gold.get(d, [])}
        P = {(e[0], e[1], e[2]) for e in pred.get(d, [])}
        for x in P & G: tp[x[2]] += 1
        for x in P - G: fp[x[2]] += 1
        for x in G - P: fn[x[2]] += 1
    def f(a, b, c):
        p = a / max(1, a + b); r = a / max(1, a + c)
        return p, r, 2 * p * r / max(1e-9, p + r)
    P_, R_, F_ = f(sum(tp.values()), sum(fp.values()), sum(fn.values()))
    per = {l: round(f(tp[l], fp[l], fn[l])[2], 4) for l in LABELS}
    present = [l for l in LABELS if tp[l] + fn[l] > 0]
    macro = sum(per[l] for l in present) / max(1, len(present))
    return {"precision": P_, "recall": R_, "f1": F_, "macro_f1": macro,
            "macro_f1_all12": sum(per.values()) / len(LABELS), "per_label_f1": per,
            "labels_in_gold": present}

Appending to /kaggle/working/src/p2_ner.py


In [31]:
# ===== Cell N12b: re-rank all saved Phase 2 models by the official metric (strict macro-F1) =====
import importlib, gc, torch, pandas as pd
import p2_peft as pe
importlib.reload(ner); importlib.reload(pm); importlib.reload(pe)
pm.HEADER, pm.DECODE = 0, "words"
CK = {"m1": "m1_dense_100m", "m2": "m2_mla_moe", "m3": "m3_encdec", "m4": "m4_copy"}
RUNS = {"m1_full_bidir": ("m1", "full"), "m1_lora": ("m1", "lora"), "m1_lorma_plus": ("m1", "lorma_plus"),
        "m1_lorma_pi": ("m1", "lorma_pi"), "m3_full": ("m3", "full"), "m2_lorma_plus": ("m2", "lorma_plus"),
        "m4_adapter": ("m4", "adapter"), "kd_student_m1x6": ("m1x6", "full"),
        "nokd_student_m1x6": ("m1x6", "full")}

def load_run(name):
    """Rebuild a saved run's architecture (backbone + method) and load its fine-tuned weights."""
    bb, method = RUNS[name]
    if bb == "m1x6":
        t = pe.shrink_dense(pm.build_tagger(f"{WORK}/checkpoints/m1_dense_100m/best.pt", bidirectional=True))
    else:
        t = pe.apply_method(pm.build_tagger(f"{WORK}/checkpoints/{CK[bb]}/best.pt", bidirectional=True), method)
    t.load_state_dict({k: v.float() for k, v in torch.load(f"{WORK}/p2_ckpt/{name}.pt").items()})
    return t.cuda().eval()

rows = {}
for name in RUNS:
    t = load_run(name)
    r = ner.strict_f1(pm.predict_docs(t, dev_docs, enc), gold)
    rows[name] = {"MACRO-F1": round(r["macro_f1"], 4), "micro-F1": round(r["f1"], 4),
                  **{l: r["per_label_f1"][l] for l in ner.LABELS}}
    del t; gc.collect(); torch.cuda.empty_cache()
print("labels present in dev gold:", r["labels_in_gold"])
display(pd.DataFrame(rows).T.sort_values("MACRO-F1", ascending=False))

labels present in dev gold: ['APP', 'RESP', 'A.COUNSEL', 'R.COUNSEL', 'JUDGE', 'AUTH', 'COURT', 'STAT', 'PREC', 'DATE', 'CASENO']


,MACRO-F1,micro-F1,APP,RESP,A.COUNSEL,R.COUNSEL,JUDGE,WIT,AUTH,COURT,STAT,PREC,DATE,CASENO
nokd_student_m1x6,0.5093,0.6116,0.1942,0.1099,0.5714,0.4000,0.6129,0.0,0.2151,0.7488,0.6867,0.3513,0.9447,0.7673
m1_full_bidir,0.4797,0.5852,0.2264,0.1136,0.5455,0.4375,0.6207,0.0,0.2126,0.7416,0.7544,0.3621,0.9521,0.3099
m4_adapter,0.4671,0.6167,0.2190,0.1304,0.2162,0.3077,0.3939,0.0,0.3158,0.8503,0.7450,0.3757,0.9521,0.6316
m1_lora,0.4552,0.6193,0.0750,0.0727,0.3415,0.2308,0.4242,0.0,0.3507,0.7629,0.7673,0.3704,0.9400,0.6719
kd_student_m1x6,0.4395,0.6146,0.0260,0.0364,0.2353,0.3529,0.3333,0.0,0.2703,0.7625,0.7621,0.3210,0.9424,0.7918
m2_lorma_plus,0.4305,0.5781,0.0896,0.1553,0.4000,0.2222,0.4308,0.0,0.3084,0.7921,0.8038,0.2891,0.9109,0.3333
m1_lorma_plus,0.4166,0.5820,0.0860,0.0385,0.2857,0.4242,0.2105,0.0,0.2871,0.7350,0.7727,0.3704,0.9303,0.4417
m1_lorma_pi,0.4009,0.5598,0.0682,0.0449,0.1714,0.3529,0.4262,0.0,0.3077,0.6817,0.7028,0.3522,0.9497,0.3524
m3_full,0.2679,0.3583,0.0638,0.0000,0.2105,0.1429,0.2857,0.0,0.0769,0.5339,0.2778,0.1165,0.8683,0.3706


In [32]:
# ===== Cell N12c: M1 + LoRA with class-weighted loss (targets macro-F1: rare labels count more) =====
import collections, json, time
cnt = collections.Counter(t for _, tags in train_ex for t in tags if t != -100)
freq = torch.tensor([cnt.get(i, 1) for i in range(len(ner.TAGS))], dtype=torch.float)
w = freq.pow(-0.5); w = w / w.mean()                              # inverse sqrt frequency, mean 1
print("weight O:", round(w[0].item(), 3), "| rarest tag weight:", round(w.max().item(), 2))
name = "m1_lora_cw"
t = pe.apply_method(pm.build_tagger(f"{WORK}/checkpoints/m1_dense_100m/best.pt", bidirectional=True), "lora")
t0 = time.time()
hist, f1 = pm.train_tagger(t, list(train_ex), dev_docs, enc, epochs=10, lr=5e-4, class_weight=w)
r = ner.strict_f1(pm.predict_docs(t, dev_docs, enc), gold)
print(f"{name}: MACRO-F1 {r['macro_f1']:.4f} | micro-F1 {r['f1']:.4f} | {r['per_label_f1']}")
results = json.load(open(RES))
results[name] = {"backbone": "m1", "method": "lora+classweight", "best_dev_f1": f1, "final": r, "hist": hist,
                 "minutes": round((time.time() - t0) / 60, 1)}
json.dump(results, open(RES, "w"), indent=1)
torch.save({k: v.half() for k, v in t.state_dict().items()}, f"{WORK}/p2_ckpt/{name}.pt")
RUNS[name] = ("m1", "lora")                                        # so load_run() can rebuild it later
del t; gc.collect(); torch.cuda.empty_cache()

weight O: 0.041 | rarest tag weight: 1.99
trainable params: 0.26M | 1513 windows | 1900 steps


/kaggle/working/src/p2_model.py:175: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scaler.step(opt); scaler.update(); sched.step(); step += 1


epoch 1: loss 1.1799 | dev strict F1 0.4104 (P 0.324 R 0.559) | 48s
epoch 2: loss 0.2543 | dev strict F1 0.5184 (P 0.440 R 0.632) | 49s
epoch 3: loss 0.1808 | dev strict F1 0.5417 (P 0.478 R 0.626) | 48s
epoch 4: loss 0.1287 | dev strict F1 0.5321 (P 0.475 R 0.605) | 49s
epoch 5: loss 0.1021 | dev strict F1 0.5638 (P 0.519 R 0.617) | 50s
epoch 6: loss 0.0843 | dev strict F1 0.5616 (P 0.512 R 0.622) | 49s
epoch 7: loss 0.0717 | dev strict F1 0.5762 (P 0.542 R 0.614) | 50s
epoch 8: loss 0.0633 | dev strict F1 0.5757 (P 0.547 R 0.608) | 50s
epoch 9: loss 0.0575 | dev strict F1 0.5677 (P 0.531 R 0.610) | 51s
epoch 10: loss 0.0542 | dev strict F1 0.5767 (P 0.546 R 0.611) | 49s
m1_lora_cw: MACRO-F1 0.5057 | micro-F1 0.5767 | {'APP': 0.2787, 'RESP': 0.1441, 'A.COUNSEL': 0.5882, 'R.COUNSEL': 0.5833, 'JUDGE': 0.5405, 'WIT': 0.0, 'AUTH': 0.2635, 'COURT': 0.8039, 'STAT': 0.7721, 'PREC': 0.273, 'DATE': 0.9024, 'CASENO': 0.4126}


## P2.8 Test predictions and submission files

The **official metric is strict macro-F1** (§8.4): the mean of per-label F1, so rare labels count as much
as frequent ones. Model selection therefore uses macro-F1 (Cell N12b), and a class-weighted loss
(inverse √frequency) was added because it directly targets this metric (+0.05 macro for LoRA).

Each submission is `prediction.json` = `{doc_id: [[start, end, label], ...]}` with **original-text character
offsets**, zipped at the archive root (§6.3). Before writing, every entity is checked: valid label,
`0 ≤ start < end ≤ len(text)`, and the document IDs exactly match the 35 test documents.

In [33]:
# ===== Cell N13: Phase 2 test predictions -> Codabench zips (prediction.json at the zip root) =====
import os, json, zipfile, gc, torch
test_docs = ner.load_fold(f"{D}/Test/fold3-text.json", None, "f3")       # 35 docs, no labels
enc_test = pm.encode_all(test_docs, tk)
SUB2 = f"{WORK}/submissions_p2"; os.makedirs(SUB2, exist_ok=True)
texts = {d["id"]: d["text"] for d in test_docs}

def write_p2(preds, name):
    """Strict format checks, then prediction.json + <name>.zip with the json at the root."""
    assert set(preds) == set(texts), "document IDs must match the test set exactly"
    out = {}
    for k, ents in preds.items():
        rows = []
        for s, e, lab in ents:
            assert lab in ner.LABELS and 0 <= s < e <= len(texts[k]), (k, s, e, lab)
            rows.append([int(s), int(e), lab])
        out[k] = rows
    d = f"{SUB2}/{name}"; os.makedirs(d, exist_ok=True)
    json.dump(out, open(f"{d}/prediction.json", "w"))
    zp = f"{d}/{name}.zip"
    with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
        z.write(f"{d}/prediction.json", arcname="prediction.json")    # no folder inside the zip
    with zipfile.ZipFile(zp) as z:
        assert z.namelist() == ["prediction.json"]
    return zp, sum(map(len, out.values()))

SUBMIT = ["m1_lora_cw", "m1_full_bidir", "m4_adapter", "m2_lorma_plus", "m1_lorma_pi",
          "kd_student_m1x6", "m3_full", "nokd_student_m1x6"]
for name in SUBMIT:
    t = load_run(name)                                                # from Cell N12b
    preds = pm.predict_docs(t, test_docs, enc_test)                   # word-consistent Viterbi
    zp, n = write_p2(preds, f"p2_{name}")
    print(f"{name:20s} -> {os.path.basename(zp)} | {n} entities ({n/len(test_docs):.0f} per doc)")
    del t; gc.collect(); torch.cuda.empty_cache()

# ---- show a few predicted entities from the first test document, as a visual sanity check ----------
k = test_docs[0]["id"]; ex = json.load(open(f"{SUB2}/p2_m1_lora_cw/prediction.json"))[k][:8]
print("\nexample doc", k, [(lab, texts[k][s:e]) for s, e, lab in ex])

m1_lora_cw           -> p2_m1_lora_cw.zip | 4929 entities (141 per doc)
m1_full_bidir        -> p2_m1_full_bidir.zip | 3800 entities (109 per doc)
m4_adapter           -> p2_m4_adapter.zip | 4412 entities (126 per doc)
m2_lorma_plus        -> p2_m2_lorma_plus.zip | 3875 entities (111 per doc)
m1_lorma_pi          -> p2_m1_lorma_pi.zip | 3882 entities (111 per doc)
kd_student_m1x6      -> p2_kd_student_m1x6.zip | 3638 entities (104 per doc)
m3_full              -> p2_m3_full.zip | 4151 entities (119 per doc)
nokd_student_m1x6    -> p2_nokd_student_m1x6.zip | 3782 entities (108 per doc)

example doc 1128835 [('COURT', 'SUPREME COURT OF INDIA'), ('CASENO', 'CIVIL APPEAL NO.6580 OF 2008'), ('CASENO', 'SPECIAL LEAVE PETITION (CIVIL) NO. 15449 OF 2004'), ('APP', 'ESSCO FABS PVT. LTD.'), ('RESP', 'STATE OF HARYANA'), ('CASENO', 'CIVIL APPEAL NO.6581 OF 2008'), ('CASENO', 'SPECIAL LEAVE PETITION (CIVIL) NO. 15544 OF 2004'), ('APP', 'PANIPAT TEACHERS')]


## N14 · Backup Phase 2 artefacts to the private HF repo

`backup_hf.py` from Phase 1 only uploaded Phase 1 folders. This cell rewrites it to also include the Phase 2 code (`src/p2_*.py`), data cache (`data_p2/`), fine-tuned weights (`p2_ckpt/`), results (`stats/p2_results.json`), the 8 test zips (`submissions_p2/`) and the M1 ablation runs (`checkpoints/abl_*`). It then runs it and lists the repo to confirm the files arrived.

`upload_folder` skips files that haven't changed, so re-running it is cheap. **Run it again after the ablations finish** (≈10:45) so their final `best.pt` and logs are saved too.

In [14]:
# ============================================================================================
# N14 — Back up Phase 1 + Phase 2 work to the private HF repo  <user>/cs781-phase1-work
#   * rewrites src/backup_hf.py with the extended allow-list (permanent: later backups use it)
#   * runs it as a subprocess; the token is passed through the environment, never printed
#   * lists the repo afterwards so we can see the Phase 2 files really arrived
# Safe to run while the ablations train on GPU 1 (upload is CPU/network only).
# ============================================================================================
import os, subprocess, textwrap
from kaggle_secrets import UserSecretsClient

BACKUP_PY = "/kaggle/working/src/backup_hf.py"

# ---- 1. new backup script (Phase 1 patterns kept, Phase 2 + ablation patterns added) -------
script = textwrap.dedent('''
    import os, sys, time
    from huggingface_hub import HfApi

    api  = HfApi(token=os.environ["HF_WRITE_TOKEN"])          # token only via env
    repo = f"{api.whoami()['name']}/cs781-phase1-work"
    api.create_repo(repo, private=True, exist_ok=True)         # no-op if it already exists

    ALLOW = [
        # ---------------- Phase 1 (unchanged) ----------------
        "src/**", "tokenizer/**", "data/**", "stats/**", "submissions/**",
        "checkpoints/*/best.pt", "checkpoints/*/config.json",
        "checkpoints/*/*.csv", "checkpoints/*/*.jsonl", "checkpoints/*/stdout.log",
        "checkpoints/m4_copy/latest.pt",
        # ---------------- Phase 2 (new) ----------------------
        "data_p2/**",            # downloaded L-NER folds + any converted external data
        "p2_ckpt/**",            # fine-tuned taggers (fp16 state dicts)
        "submissions_p2/**",     # the 8 test zips + prediction.json
        # ---------------- M1 ablations (new) -----------------
        "checkpoints/abl_*/*.log", "checkpoints/abl_*/*.txt",   # (best.pt/csv/jsonl already covered above)
    ]
    IGNORE = ["**/__pycache__/**", "**/*.tmp", "**/.ipynb_checkpoints/**"]

    t0 = time.time()
    try:
        api.upload_folder(repo_id=repo, folder_path="/kaggle/working",
                          allow_patterns=ALLOW, ignore_patterns=IGNORE,
                          commit_message=f"backup {time.strftime('%Y-%m-%d %H:%M')}")
        print(f"BACKUP DONE -> {repo} in {time.time()-t0:.0f}s")
    except Exception as e:
        print("BACKUP FAILED:", repr(e)); sys.exit(1)
''')
with open(BACKUP_PY, "w") as f:
    f.write(script)
print("rewrote", BACKUP_PY)

# ---- 2. what is about to be uploaded (sizes, so a 5-10 min upload isn't a surprise) ---------
def du(path):
    tot = 0
    for root, _, files in os.walk(path):
        tot += sum(os.path.getsize(os.path.join(root, x)) for x in files)
    return tot / 1e6
for p in ["src", "data_p2", "p2_ckpt", "submissions_p2", "stats"]:
    full = f"/kaggle/working/{p}"
    print(f"  {p:<16} {du(full):9.1f} MB" if os.path.exists(full) else f"  {p:<16}   (missing)")

# ---- 3. run it (token via env only) --------------------------------------------------------
env = {**os.environ, "HF_WRITE_TOKEN": UserSecretsClient().get_secret("HF_WRITE_TOKEN")}
r = subprocess.run(f"python {BACKUP_PY}", shell=True, capture_output=True, text=True, env=env)
print(r.stdout[-2000:])
if r.returncode != 0:
    print(r.stderr[-2000:])                                    # show the error if it failed

# ---- 4. verify: count Phase 2 files now on the hub ------------------------------------------
from huggingface_hub import HfApi
api   = HfApi(token=env["HF_WRITE_TOKEN"])
repo  = f"{api.whoami()['name']}/cs781-phase1-work"
files = api.list_repo_files(repo)
for pre in ["src/p2_", "data_p2/", "p2_ckpt/", "submissions_p2/", "stats/p2_results", "checkpoints/abl_"]:
    hit = [x for x in files if x.startswith(pre)]
    print(f"  {pre:<20} {len(hit):3d} files", ("  e.g. " + hit[0]) if hit else "")

rewrote /kaggle/working/src/backup_hf.py
  src                    0.3 MB
  data_p2               22.4 MB
  p2_ckpt             6606.9 MB
  submissions_p2         2.5 MB
  stats                373.7 MB
BACKUP DONE -> Utkarsh-1718-1/cs781-phase1-work in 86s

  src/p2_                3 files   e.g. src/p2_model.py
  data_p2/              15 files   e.g. data_p2/Phase_2_LNER/Test/fold3-text.json
  p2_ckpt/              26 files   e.g. p2_ckpt/abl_base_lora_cw.pt
  submissions_p2/       36 files   e.g. submissions_p2/p2_abl_base_lora_cw/p2_abl_base_lora_cw.zip
  stats/p2_results       1 files   e.g. stats/p2_results.json
  checkpoints/abl_      15 files   e.g. checkpoints/abl_base/best.pt


## P2.9 · External data: InLegalNER (Kalamkar et al., 2022)

**Why:** we have only 70 annotated training documents. InLegalNER (opennyaiorg, MIT licence) adds about 9.4k annotated judgment sentences and about 1.5k annotated preambles from Indian court judgments, which is the same domain as ours. External data is allowed (Discord announcement) and must be described in the report.

**Label mapping (InLegalNER → ours):**

| InLegalNER | ours | note |
|---|---|---|
| PETITIONER / RESPONDENT | APP / RESP | party roles, our hardest classes |
| JUDGE, COURT, DATE | same | |
| WITNESS / STATUTE / PRECEDENT / CASE_NUMBER | WIT / STAT / PREC / CASENO | |
| LAWYER, ORG, PROVISION | *ignored* (loss = −100) | we can't tell A.COUNSEL from R.COUNSEL; ORG may be our AUTH; PROVISION may be inside our STAT |
| GPE, OTHER_PERSON | O | these are not entities in our scheme |

**Leakage control:** any InLegalNER record that overlaps text from our **test fold** (matched on word 8-grams) is dropped.

**Plan:** (N10) download, convert and inspect → (N10b) intermediate training on InLegalNER, then fine-tune on our 70 docs → compare dev strict macro-F1 with the same model trained without the external data (m1_lora_cw 0.506).

In [36]:
# ============================================================================================
# N10 — InLegalNER: download, convert to our {text, ents, ignore} format, leakage check, inspect
#   Output: /kaggle/working/data_p2/inlegalner_conv.json
#   CPU only — safe while the M1 ablations train on GPU 1.
# ============================================================================================
import os, re, glob, json, random, hashlib
from collections import Counter, defaultdict
import numpy as np, pandas as pd
from huggingface_hub import snapshot_download
from kaggle_secrets import UserSecretsClient

EXT_DIR = "/kaggle/working/data_p2/inlegalner"
OUT     = "/kaggle/working/data_p2/inlegalner_conv.json"
TOKEN   = UserSecretsClient().get_secret("HF_WRITE_TOKEN")     # HF_TOKEN gave 401 on this account

# ---- 1. download (gated: terms must be accepted on the HF page first) -----------------------
try:
    snapshot_download("opennyaiorg/InLegalNER", repo_type="dataset", local_dir=EXT_DIR, token=TOKEN)
except Exception as e:
    raise SystemExit(f"DOWNLOAD FAILED ({type(e).__name__}). Accept the dataset terms on "
                     f"huggingface.co/datasets/opennyaiorg/InLegalNER, then re-run.\n{e}")
files = sorted(glob.glob(f"{EXT_DIR}/**/*.parquet", recursive=True) +
               glob.glob(f"{EXT_DIR}/**/*.json*", recursive=True))
print("files:"); [print("  ", os.path.relpath(f, EXT_DIR), f"{os.path.getsize(f)/1e6:.2f} MB") for f in files]

# ---- 2. load every split into one list of raw records --------------------------------------
raw = []
for f in files:
    split = "dev" if re.search(r"dev|valid", f, re.I) else "test" if re.search(r"test", f, re.I) else "train"
    if f.endswith(".parquet"):
        recs = pd.read_parquet(f).to_dict("records")
    elif f.endswith(".jsonl"):
        recs = [json.loads(l) for l in open(f)]
    else:
        try:
            j = json.load(open(f)); recs = j if isinstance(j, list) else []
        except Exception:
            recs = []
    for r in recs:
        raw.append((split, r))
print(f"\nraw records: {len(raw)}  by split: {Counter(s for s, _ in raw)}")
print("columns of first record:", list(raw[0][1].keys()))
print("first record (truncated):", repr(raw[0][1])[:1200])

# ---- 3. generic extractors (handles Label-Studio style {data:{text}, annotations:[{result:[{value:{start,end,labels}}]}]}) ----
def find_text(r):
    if isinstance(r.get("data"), dict) and "text" in r["data"]:
        return r["data"]["text"]
    return r.get("text")

def find_spans(r):
    out = set()
    def walk(o):
        if isinstance(o, dict):
            if "start" in o and "end" in o and "labels" in o:
                lab = o["labels"]
                lab = lab[0] if isinstance(lab, (list, tuple, np.ndarray)) and len(lab) else lab
                out.add((int(o["start"]), int(o["end"]), str(lab)))
            for v in o.values():
                walk(v)
        elif isinstance(o, (list, tuple, np.ndarray)):
            for v in o:
                walk(v)
    walk(r)
    return sorted(out)

# ---- 4. label mapping ------------------------------------------------------------------------
MAP    = {"PETITIONER": "APP", "RESPONDENT": "RESP", "JUDGE": "JUDGE", "COURT": "COURT", "DATE": "DATE",
          "WITNESS": "WIT", "STATUTE": "STAT", "PRECEDENT": "PREC", "CASE_NUMBER": "CASENO"}
IGNORE = {"LAWYER", "ORG", "PROVISION"}          # tokens inside -> loss ignored (-100)
TO_O   = {"GPE", "OTHER_PERSON"}                 # simply not entities for us

conv, bad_off, unknown, raw_lab = [], 0, Counter(), Counter()
for k, (split, r) in enumerate(raw):
    text = find_text(r)
    if not isinstance(text, str) or not text.strip():
        continue
    spans = find_spans(r)
    if split == "test" and not spans:
        continue                                  # unlabelled test split -> useless for training
    ents, ign = [], []
    for s, e, lab in spans:
        raw_lab[lab] += 1
        if not (0 <= s < e <= len(text)):
            bad_off += 1; continue
        if lab in MAP:      ents.append([s, e, MAP[lab]])
        elif lab in IGNORE: ign.append([s, e])
        elif lab in TO_O:   pass
        else:               unknown[lab] += 1
    meta = str(r.get("meta", "")) + str(r.get("source", ""))
    kind = "preamble" if ("preamble" in meta.lower() or any(l == "LAWYER" for *_, l in spans)) else "judgement"
    conv.append({"id": f"ext{k}", "text": text, "ents": ents, "ignore": ign,
                 "split": split, "kind": kind, "fold": "ext"})
print(f"\nconverted: {len(conv)} records | bad offsets {bad_off} | unknown labels {dict(unknown)}")
print("raw label counts:", dict(raw_lab.most_common()))
print("mapped label counts:", dict(Counter(e[2] for d in conv for e in d["ents"]).most_common()))
print("kinds:", Counter(d["kind"] for d in conv), "| median chars:",
      {kd: int(np.median([len(d["text"]) for d in conv if d["kind"] == kd])) for kd in ("preamble", "judgement")})

# ---- 5. our own data (train folds + test fold), for leakage check and side-by-side ----------
ours = []
for tp in sorted(glob.glob("/kaggle/working/data_p2/Phase_2_LNER/**/*text*.json", recursive=True)):
    gp = next((tp.replace("text", x) for x in ("gold", "labels", "label", "ann")
               if os.path.exists(tp.replace("text", x))), None)
    is_test = "/Test/" in tp
    T = json.load(open(tp)); G = json.load(open(gp)) if gp else {}
    for i, t in T.items():
        ours.append({"id": i, "text": t, "ents": G.get(i, []), "test": is_test})
print(f"\nour docs: {sum(not d['test'] for d in ours)} train, {sum(d['test'] for d in ours)} test")

def shingles(t, n=8):
    w = re.findall(r"[a-z0-9]+", t.lower())
    return {hashlib.md5(" ".join(w[i:i + n]).encode()).digest()[:8] for i in range(max(0, len(w) - n + 1))}

S_test  = set().union(*[shingles(d["text"]) for d in ours if d["test"]])
S_train = set().union(*[shingles(d["text"]) for d in ours if not d["test"]])
for d in conv:
    sh = shingles(d["text"])
    d["olap_test"]  = len(sh & S_test)  / max(1, len(sh))       # fraction of its 8-grams found in our test docs
    d["olap_train"] = len(sh & S_train) / max(1, len(sh))
olap = [d for d in conv if d["olap_test"] >= 0.3]
print(f"overlap with OUR TEST fold (>=30% of 8-grams): {len(olap)} records -> dropped")
print(f"overlap with our train folds (>=30%):          {sum(d['olap_train'] >= 0.3 for d in conv)} (kept; just noted)")
for d in olap[:3]:
    print("   e.g.", repr(d["text"][:150]))
conv = [d for d in conv if d["olap_test"] < 0.3]

# ---- 6. side-by-side: annotation conventions (span boundaries matter for STRICT F1) ----------
random.seed(0)
by_ext, by_our = defaultdict(list), defaultdict(list)
for d in conv:
    for s, e, l in d["ents"]: by_ext[l].append(d["text"][s:e])
for d in ours:
    if not d["test"]:
        for s, e, l, *_ in d["ents"]: by_our[l].append(d["text"][s:e])
for lab in ["APP", "RESP", "JUDGE", "COURT", "STAT", "PREC", "CASENO", "DATE", "WIT"]:
    print(f"\n--- {lab}:  ours n={len(by_our[lab])}  | InLegalNER n={len(by_ext[lab])}")
    print("   ours:", [x[:60] for x in random.sample(by_our[lab], min(4, len(by_our[lab])))])
    print("   ext :", [x[:60] for x in random.sample(by_ext[lab], min(4, len(by_ext[lab])))])

# ---- 7. save ---------------------------------------------------------------------------------
json.dump(conv, open(OUT, "w"))
print(f"\nsaved {len(conv)} records -> {OUT}  ({os.path.getsize(OUT)/1e6:.1f} MB)")
print("split x kind:", Counter((d["split"], d["kind"]) for d in conv))

Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

files:
   data/dev-00000-of-00001-b148266485fd7aeb.parquet 0.39 MB
   data/test-00000-of-00001-814a730def5e8488.parquet 1.52 MB
   data/train-00000-of-00001-de25c1ae1db42f79.parquet 3.53 MB

raw records: 16570  by split: Counter({'train': 10995, 'test': 4501, 'dev': 1074})
columns of first record: ['annotations', 'meta', 'id', 'data']
first record (truncated): {'annotations': array([{'result': array([{'from_name': 'label', 'id': '25TFDATV', 'to_name': 'text', 'type': 'labels', 'value': {'end': 22, 'labels': array(['STATUTE'], dtype=object), 'start': 10, 'text': 'Constitution'}},
              {'from_name': 'label', 'id': 'F706LMQM', 'to_name': 'text', 'type': 'labels', 'value': {'end': 155, 'labels': array(['PRECEDENT'], dtype=object), 'start': 108, 'text': 'R.C. Cooper v. Union of India, (1970) 1 SCC 248'}},
              {'from_name': 'label', 'id': '2EURBJSZ', 'to_name': 'text', 'type': 'labels', 'value': {'end': 209, 'labels': array(['PRECEDENT'], dtype=object), 'start': 160, 'text

### N15b · Two-stage training: InLegalNER → our data (plus a same-code control)

* **Packing:** judgment sentences are short (median about 190 characters), so they are shuffled and packed into pseudo-documents of about 1,800 characters (≈450 tokens). Each preamble (median about 920 characters) stays as its own document, because it looks like the start of our documents, where APP/RESP/JUDGE/COURT appear.
* **Ignored spans:** tokens inside LAWYER/ORG/PROVISION spans, and pure-space tokens, get label −100 (no loss).
* **Dev hygiene:** external records whose word 8-grams overlap our 15 dev documents by 30% or more are also dropped.
* **Runs** (same backbone M1, LoRA r=8 on q and v, class-weighted CE, word-Viterbi; best epoch chosen by **dev strict macro-F1**, the official metric):
  1. **Stage A**: train on InLegalNER only, then evaluate on our dev set (zero-shot transfer).
  2. **Stage B**: continue the same LoRA and head on our 55 training documents → `m1_lora_cw_ext`.
  3. **Control**: same code, same settings, no external data → `m1_lora_cw_ctrl`. This is the fair baseline for measuring the gain.

In [38]:
# ============================================================================================
# N15b — InLegalNER intermediate training (Stage A) -> fine-tune on our data (Stage B) + control
#   GPU 0 only (the M1 ablations keep GPU 1). Roughly 30-45 min in total.
#   Outputs: p2_ckpt/m1_lora_cw_ext.pt, p2_ckpt/m1_lora_cw_ctrl.pt, stats/p2_ext_results.json
# ============================================================================================
import os, re, glob, json, time, random, hashlib, sys
from collections import Counter
import numpy as np, torch
sys.path.insert(0, "/kaggle/working/src")
import p2_ner as ner, p2_model as pm, p2_peft as pp

DEV = "cuda:0"; torch.cuda.set_device(0)
pm.HEADER = 0                                   # header context was dropped (N-cells) -> keep it off
BIDIR, METHOD = False, "lora"                   # same setting as m1_lora_cw (RUNS = ("m1","lora"))
LR, HEAD_LR, BS = 5e-4, 1e-3, 8                 # LoRA lr / head lr / batch
EP_A, EP_B = 2, 8                               # epochs: Stage A (external), Stage B (ours) and control
WORK = "/kaggle/working"

# ---- 1. tokenizer + M1 checkpoint (reuse notebook objects when they exist) -----------------
if "tk" not in globals():
    from tokenizers import Tokenizer
    tk = Tokenizer.from_file(glob.glob(f"{WORK}/tokenizer/*.json")[0])
# CK maps model -> run NAME; turn it into the checkpoint PATH (falls back to a glob)
_m1 = CK["m1"] if "CK" in globals() and "m1" in CK else None
M1_CKPT = (_m1 if _m1 and os.path.isfile(_m1) else
           f"{WORK}/checkpoints/{_m1}/best.pt" if _m1 else
           sorted(glob.glob(f"{WORK}/checkpoints/m1*/best.pt"))[0])
assert os.path.isfile(M1_CKPT), M1_CKPT
print("M1 checkpoint:", M1_CKPT)

# ---- 2. our data: same folds + same deterministic dev split as before ------------------------
docs = []
for tp in sorted(glob.glob(f"{WORK}/data_p2/Phase_2_LNER/**/*text*.json", recursive=True)):
    if "/Test/" in tp:
        continue
    gp = next(tp.replace("text", x) for x in ("gold", "labels", "label", "ann")
              if os.path.exists(tp.replace("text", x)))
    docs += ner.load_fold(tp, gp, fold=os.path.basename(tp).split("-")[0])
train_docs, dev_docs = ner.dev_split(docs, 5)
print(f"ours: {len(train_docs)} train / {len(dev_docs)} dev docs")
if "dev_docs_prev" in globals():                 # optional sanity check against an older split
    assert {d['id'] for d in dev_docs} == {d['id'] for d in dev_docs_prev}

# ---- 3. external data: drop dev-overlapping records, pack, encode with ignore mask -----------
ext = json.load(open(f"{WORK}/data_p2/inlegalner_conv.json"))
def shingles(t, n=8):
    w = re.findall(r"[a-z0-9]+", t.lower())
    return {hashlib.md5(" ".join(w[i:i + n]).encode()).digest()[:8] for i in range(max(0, len(w) - n + 1))}
S_dev = set().union(*[shingles(d["text"]) for d in dev_docs])
n0 = len(ext)
ext = [r for r in ext if len(shingles(r["text"]) & S_dev) / max(1, len(shingles(r["text"]))) < 0.3]
print(f"external: {n0} -> {len(ext)} after dropping dev-overlapping records")

def pack(recs, max_chars=1800):
    """Concatenate short records ('\\n\\n' between them) into pseudo-documents, shifting offsets."""
    out, cur = [], None
    for r in recs:
        if cur and len(cur["text"]) + 2 + len(r["text"]) > max_chars:
            out.append(cur); cur = None
        if cur is None:
            cur = {"id": f"pk{len(out)}", "text": "", "ents": [], "ignore": []}
        off = len(cur["text"]) + (2 if cur["text"] else 0)
        cur["text"] += ("\n\n" if cur["text"] else "") + r["text"]
        cur["ents"]   += [[s + off, e + off, l] for s, e, l in r["ents"]]
        cur["ignore"] += [[s + off, e + off] for s, e in r["ignore"]]
    return out + ([cur] if cur else [])

random.seed(0)
judg = [r for r in ext if r["kind"] == "judgement"]; random.shuffle(judg)
pre  = [dict(r, id=f"pre{i}") for i, r in enumerate(r for r in ext if r["kind"] == "preamble")]
ext_docs = pack(judg) + pre
print(f"external pseudo-docs: {len(ext_docs)} ({len(ext_docs)-len(pre)} packed judgement + {len(pre)} preambles)")

def encode_with_ignore(d):
    ids, spans = ner.encode_doc(d["text"], tk)
    tags = ner.bio_tags(spans, d["ents"])
    bad = np.zeros(len(d["text"]) + 1, dtype=bool)
    for s, e in d.get("ignore", []):
        bad[s:e] = True                                            # LAWYER / ORG / PROVISION chars
    tags = [-100 if sp is None or bad[sp[0]:sp[1]].any() else t for t, sp in zip(tags, spans)]
    return ids, spans, tags

ENC = pm.encode_all(train_docs + dev_docs, tk)                     # ours (space tokens -> -100)
ENC.update({d["id"]: encode_with_ignore(d) for d in ext_docs})
ext_ex  = pm.make_examples(ENC, [d["id"] for d in ext_docs], header=0)
ours_ex = pm.make_examples(ENC, [d["id"] for d in train_docs], header=0)
print(f"windows: external {len(ext_ex)} | ours {len(ours_ex)}")
print("external tag counts:", Counter(ner.TAGS[t][2:] for _, ts in ext_ex for t in ts
                                      if t > 0 and ner.TAGS[t].startswith("B-")).most_common())

# ---- 4. class weights (inverse sqrt frequency on OUR training tags, as for m1_lora_cw) -------
cnt = np.bincount([t for _, ts in ours_ex for t in ts if t >= 0], minlength=len(ner.TAGS)).astype(float)
CW = torch.tensor(np.where(cnt > 0, np.sqrt(cnt.max() / np.maximum(cnt, 1)), 1.0), dtype=torch.float)
CW = CW.clamp(max=10.0)                                            # cap rare-tag weights
print("class weight range:", round(CW.min().item(), 2), "-", round(CW.max().item(), 2))

# ---- 5. select epochs by STRICT MACRO-F1 (official metric) -----------------------------------
_strict = ner.strict_f1
def macro_of(r, gold):
    present = {e[2] for es in gold.values() for e in es}           # labels that occur in dev gold
    return float(np.mean([r["per_label_f1"][l] for l in ner.LABELS if l in present]))
def strict_f1_macro(pred, gold):                                    # train_tagger reads r["f1"]
    r = _strict(pred, gold); r["micro"] = r["f1"]; r["f1"] = macro_of(r, gold); return r

def evaluate(model):
    gold = {d["id"]: d["ents"] for d in dev_docs}
    r = _strict(pm.predict_docs(model, dev_docs, ENC, device=DEV), gold)
    return {"macro": round(macro_of(r, gold), 4), "micro": round(r["f1"], 4), "per_label": r["per_label_f1"]}

def new_model():
    return pp.apply_method(pm.build_tagger(M1_CKPT, bidirectional=BIDIR), METHOD)

def save(model, name):
    torch.save({k: v.half() for k, v in model.state_dict().items()}, f"{WORK}/p2_ckpt/{name}.pt")
    if "RUNS" in globals():
        RUNS[name] = ("m1", METHOD)                                # so load_run(name) works later

OUT = {}
ner.strict_f1 = strict_f1_macro
try:
    # ---- Stage A: external data only ----------------------------------------------------------
    t0 = time.time(); model = new_model()
    print(f"\n=== Stage A: InLegalNER ({EP_A} ep) ===  (log 'dev strict F1' = MACRO)")
    hA, _ = pm.train_tagger(model, ext_ex, dev_docs, ENC, epochs=EP_A, lr=LR, head_lr=HEAD_LR,
                            bs=BS, class_weight=CW, device=DEV)
    OUT["stageA_zero_shot"] = {**evaluate(model), "min": round((time.time() - t0) / 60, 1)}
    print("Stage A on our dev:", {k: v for k, v in OUT["stageA_zero_shot"].items() if k != "per_label"})

    # ---- Stage B: continue on our data ---------------------------------------------------------
    t0 = time.time()
    print(f"\n=== Stage B: our data ({EP_B} ep), continuing Stage A weights ===")
    hB, _ = pm.train_tagger(model, ours_ex, dev_docs, ENC, epochs=EP_B, lr=LR, head_lr=HEAD_LR,
                            bs=BS, class_weight=CW, device=DEV)
    OUT["m1_lora_cw_ext"] = {**evaluate(model), "min": round((time.time() - t0) / 60, 1)}
    save(model, "m1_lora_cw_ext"); del model; torch.cuda.empty_cache()

    # ---- Control: identical code, no external data ----------------------------------------------
    t0 = time.time(); model = new_model()
    print(f"\n=== Control: our data only ({EP_B} ep) ===")
    hC, _ = pm.train_tagger(model, ours_ex, dev_docs, ENC, epochs=EP_B, lr=LR, head_lr=HEAD_LR,
                            bs=BS, class_weight=CW, device=DEV)
    OUT["m1_lora_cw_ctrl"] = {**evaluate(model), "min": round((time.time() - t0) / 60, 1)}
    save(model, "m1_lora_cw_ctrl"); del model; torch.cuda.empty_cache()
finally:
    ner.strict_f1 = _strict                                        # always restore the original metric

# ---- 6. summary ------------------------------------------------------------------------------
json.dump(OUT, open(f"{WORK}/stats/p2_ext_results.json", "w"), indent=1)
print(f"\n{'run':<22}{'macro':>8}{'micro':>8}{'min':>6}")
for k, v in OUT.items():
    print(f"{k:<22}{v['macro']:>8.4f}{v['micro']:>8.4f}{v['min']:>6}")
print("\nper-label F1 (ctrl -> ext):")
for l in ner.LABELS:
    a, b = OUT["m1_lora_cw_ctrl"]["per_label"][l], OUT["m1_lora_cw_ext"]["per_label"][l]
    print(f"  {l:<10} {a:.3f} -> {b:.3f}  {'+' if b > a else ' '}{b - a:+.3f}")

M1 checkpoint: /kaggle/working/checkpoints/m1_dense_100m/best.pt
ours: 60 train / 10 dev docs
external: 15980 -> 15974 after dropping dev-overlapping records
external pseudo-docs: 3781 (1668 packed judgement + 2113 preambles)
windows: external 4232 | ours 1513
external tag counts: [('RESP', 6653), ('APP', 4780), ('COURT', 3892), ('DATE', 3216), ('JUDGE', 3211), ('STAT', 2993), ('PREC', 2173), ('CASENO', 1828), ('WIT', 1351)]
class weight range: 1.0 - 10.0

=== Stage A: InLegalNER (2 ep) ===  (log 'dev strict F1' = MACRO)
trainable params: 0.26M | 4232 windows | 1058 steps


/kaggle/working/src/p2_model.py:175: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scaler.step(opt); scaler.update(); sched.step(); step += 1


epoch 1: loss 0.5572 | dev strict F1 0.2598 (P 0.340 R 0.483) | 112s
epoch 2: loss 0.2342 | dev strict F1 0.2978 (P 0.401 R 0.514) | 116s
Stage A on our dev: {'macro': 0.2978, 'micro': 0.4507, 'min': 3.9}

=== Stage B: our data (8 ep), continuing Stage A weights ===
trainable params: 0.26M | 1513 windows | 1520 steps
epoch 1: loss 0.4046 | dev strict F1 0.4373 (P 0.480 R 0.609) | 48s
epoch 2: loss 0.1924 | dev strict F1 0.4328 (P 0.501 R 0.621) | 48s
epoch 3: loss 0.1536 | dev strict F1 0.4796 (P 0.522 R 0.614) | 48s
epoch 4: loss 0.1248 | dev strict F1 0.4956 (P 0.528 R 0.636) | 48s
epoch 5: loss 0.1082 | dev strict F1 0.4934 (P 0.518 R 0.626) | 48s
epoch 6: loss 0.0949 | dev strict F1 0.4946 (P 0.543 R 0.626) | 47s
epoch 7: loss 0.0864 | dev strict F1 0.5080 (P 0.543 R 0.636) | 49s
epoch 8: loss 0.0804 | dev strict F1 0.5040 (P 0.537 R 0.633) | 46s

=== Control: our data only (8 ep) ===
trainable params: 0.26M | 1513 windows | 1520 steps
epoch 1: loss 0.9275 | dev strict F1 0.3980 (P

### N15c–N15d · Test zip + robustness check of the external-data gain

**N15b result:** the external data gave a small macro gain (0.508 vs 0.495 for the control), within dev noise (10 docs). The model reached a good score earlier (epoch 1: 0.437 vs 0.398), and zero-shot transfer gave micro 0.45.

* **N15c**: writes the test submission for `m1_lora_cw_ext`, in the same format as the earlier zips.
* **N15d**:
  1. **Seeds:** Stage A is run once, then Stage B and the control are each re-run with seeds 1 and 2. The table reports mean ± std over 3 seeds, so the claim no longer rests on a single seed.
  2. **Capacity:** the same two-stage recipe with **full fine-tuning plus bidirectional attention** (100M trainable parameters instead of 0.26M), against its own control. Hypothesis: a bigger trainable set can absorb more from 4.2k external windows.

In [39]:
# ============================================================================================
# N15c — Test submission for m1_lora_cw_ext (format copied from an existing Phase-2 zip)
#   Uses objects from N15b (tk, ENC, new_model, DEV, WORK). About 1 min on GPU 0.
# ============================================================================================
import zipfile

def write_submission(model, name):
    # ---- test docs (fold 3) -----------------------------------------------------------------
    tp = glob.glob(f"{WORK}/data_p2/Phase_2_LNER/Test/*text*.json")[0]
    T = json.load(open(tp))
    test_docs = [{"id": str(k), "text": v, "ents": []} for k, v in T.items()]
    ENC.update(pm.encode_all(test_docs, tk))
    pred = pm.predict_docs(model, test_docs, ENC, device=DEV)

    # ---- mirror the structure of a zip that already scored on Codabench -----------------------
    ref = json.load(open(f"{WORK}/submissions_p2/p2_m1_lora_cw/prediction.json"))
    assert set(map(str, ref)) == set(T), "test ids differ from the reference submission"
    ex = next(e for v in ref.values() for e in v)               # one reference entity
    text = {d["id"]: d["text"] for d in test_docs}
    def fmt(doc, s, e, l):
        if isinstance(ex, dict):                                # e.g. {"start":..,"end":..,"label":..}
            ks = list(ex)
            return {ks[0]: s, ks[1]: e, ks[2]: l, **({ks[3]: text[doc][s:e]} if len(ks) > 3 else {})}
        return [s, e, l] + ([text[doc][s:e]] if len(ex) > 3 else [])
    out = {d: [fmt(d, s, e, l) for s, e, l in ents] for d, ents in pred.items()}

    # ---- write zip with prediction.json at the root -----------------------------------------
    folder = f"{WORK}/submissions_p2/p2_{name}"; os.makedirs(folder, exist_ok=True)
    json.dump(out, open(f"{folder}/prediction.json", "w"))
    with zipfile.ZipFile(f"{folder}/p2_{name}.zip", "w", zipfile.ZIP_DEFLATED) as z:
        z.write(f"{folder}/prediction.json", arcname="prediction.json")
    n = sum(len(v) for v in out.values())
    print(f"{name}: {n} entities ({n/len(out):.0f}/doc) | label mix {Counter(e[2] for v in pred.values() for e in v).most_common(5)}")
    print("   reference entity format:", ex, "| ours:", next(e for v in out.values() for e in v))
    return f"{folder}/p2_{name}.zip"

def load_saved(name, bidir=BIDIR, method=METHOD):
    m = pp.apply_method(pm.build_tagger(M1_CKPT, bidirectional=bidir), method)
    sd = torch.load(f"{WORK}/p2_ckpt/{name}.pt", map_location="cpu")
    m.load_state_dict({k: v.float() for k, v in sd.items()})
    return m.to(DEV)

m = load_saved("m1_lora_cw_ext")
print("dev check (should be 0.508 macro):", {k: v for k, v in evaluate(m).items() if k != "per_label"})
print(write_submission(m, "m1_lora_cw_ext"))
del m; torch.cuda.empty_cache()

dev check (should be 0.508 macro): {'macro': 0.508, 'micro': 0.5861}
m1_lora_cw_ext: 5183 entities (148/doc) | label mix [('STAT', 1020), ('DATE', 779), ('PREC', 733), ('COURT', 624), ('AUTH', 596)]
   reference entity format: [18, 40, 'COURT'] | ours: [18, 40, 'COURT']
/kaggle/working/submissions_p2/p2_m1_lora_cw_ext/p2_m1_lora_cw_ext.zip


In [41]:
# ============================================================================================
# N15d — (1) 2 extra seeds for ext vs control (LoRA)   (2) full fine-tuning + bidirectional, ext vs control
#   Reuses N15b objects (ext_ex, ours_ex, ENC, CW, evaluate, strict_f1_macro, _strict, ...).
#   Results are appended to stats/p2_ext_results.json; the full-FT runs are saved to p2_ckpt/.
# ============================================================================================
def make(bidir, method):
    return pp.apply_method(pm.build_tagger(M1_CKPT, bidirectional=bidir), method)

def run(tag, bidir, method, lr, seed, init_state=None, data=None, epochs=EP_B):
    """One training run on `data` (default: our windows); optional starting weights."""
    t0 = time.time(); m = make(bidir, method)
    if init_state is not None:
        m.load_state_dict(init_state)
    print(f"\n=== {tag} (seed {seed}) ===")
    pm.train_tagger(m, data or ours_ex, dev_docs, ENC, epochs=epochs, lr=lr, head_lr=HEAD_LR,
                    bs=BS, class_weight=CW, device=DEV, seed=seed)
    r = {**evaluate(m), "min": round((time.time() - t0) / 60, 1)}
    print(f"   -> macro {r['macro']:.4f} | micro {r['micro']:.4f}")
    return m, r

OUT = json.load(open(f"{WORK}/stats/p2_ext_results.json"))
ner.strict_f1 = strict_f1_macro                                    # select epochs by macro-F1
try:
    # ---- (1) seeds for the LoRA comparison (seed 0 = N15b) -----------------------------------
    mA, rA = run("LoRA Stage A (ext)", BIDIR, METHOD, LR, seed=0, data=ext_ex, epochs=EP_A)
    SA = {k: v.detach().cpu().clone() for k, v in mA.state_dict().items()}; del mA
    for s in (1, 2):
        m, OUT[f"m1_lora_cw_ext_s{s}"]  = run("LoRA ext->ours", BIDIR, METHOD, LR, s, init_state=SA); del m
        m, OUT[f"m1_lora_cw_ctrl_s{s}"] = run("LoRA control",   BIDIR, METHOD, LR, s);               del m
        torch.cuda.empty_cache()
    json.dump(OUT, open(f"{WORK}/stats/p2_ext_results.json", "w"), indent=1)   # checkpoint results

    # ---- (2) capacity: full fine-tuning + bidirectional, ext vs control ----------------------
    FT_LR = 5e-5                                                    # full-FT lr (train_tagger default)
    mA, OUT["full_bidir_stageA"] = run("FULL+bidir Stage A (ext)", True, "full", FT_LR, 0,
                                       data=ext_ex, epochs=EP_A)
    SA = {k: v.detach().cpu().clone() for k, v in mA.state_dict().items()}; del mA; torch.cuda.empty_cache()
    m, OUT["m1_full_bidir_ext"]  = run("FULL+bidir ext->ours", True, "full", FT_LR, 0, init_state=SA)
    torch.save({k: v.half() for k, v in m.state_dict().items()}, f"{WORK}/p2_ckpt/m1_full_bidir_ext.pt")
    del m, SA; torch.cuda.empty_cache()
    m, OUT["m1_full_bidir_ctrl"] = run("FULL+bidir control", True, "full", FT_LR, 0)
    torch.save({k: v.half() for k, v in m.state_dict().items()}, f"{WORK}/p2_ckpt/m1_full_bidir_ctrl.pt")
    del m; torch.cuda.empty_cache()
finally:
    ner.strict_f1 = _strict
    json.dump(OUT, open(f"{WORK}/stats/p2_ext_results.json", "w"), indent=1)

# ---- summary: mean ± std over 3 seeds (LoRA), single seed (full FT) ----------------------------
def ms(keys, f):
    v = np.array([OUT[k][f] for k in keys]); return f"{v.mean():.4f} ± {v.std(ddof=1):.4f}"
ext_k  = ["m1_lora_cw_ext",  "m1_lora_cw_ext_s1",  "m1_lora_cw_ext_s2"]
ctrl_k = ["m1_lora_cw_ctrl", "m1_lora_cw_ctrl_s1", "m1_lora_cw_ctrl_s2"]
print(f"\n{'setting':<26}{'macro (3 seeds)':>20}{'micro (3 seeds)':>20}")
print(f"{'LoRA  ext -> ours':<26}{ms(ext_k,'macro'):>20}{ms(ext_k,'micro'):>20}")
print(f"{'LoRA  control':<26}{ms(ctrl_k,'macro'):>20}{ms(ctrl_k,'micro'):>20}")
gain = [OUT[a]["macro"] - OUT[b]["macro"] for a, b in zip(ext_k, ctrl_k)]
print(f"paired macro gain per seed: {[round(g, 4) for g in gain]}  (all > 0 -> consistent)")
for k in ["full_bidir_stageA", "m1_full_bidir_ext", "m1_full_bidir_ctrl"]:
    print(f"{k:<26}{OUT[k]['macro']:>20.4f}{OUT[k]['micro']:>20.4f}")


=== LoRA Stage A (ext) (seed 0) ===
trainable params: 0.26M | 4232 windows | 1058 steps
epoch 1: loss 0.5487 | dev strict F1 0.3211 (P 0.403 R 0.548) | 113s
epoch 2: loss 0.2301 | dev strict F1 0.3257 (P 0.429 R 0.536) | 116s
   -> macro 0.3257 | micro 0.4764

=== LoRA ext->ours (seed 1) ===
trainable params: 0.26M | 1513 windows | 1520 steps
epoch 1: loss 0.4024 | dev strict F1 0.4503 (P 0.480 R 0.638) | 46s
epoch 2: loss 0.1904 | dev strict F1 0.4869 (P 0.463 R 0.664) | 48s
epoch 3: loss 0.1481 | dev strict F1 0.4695 (P 0.480 R 0.659) | 48s
epoch 4: loss 0.1225 | dev strict F1 0.5087 (P 0.527 R 0.668) | 48s
epoch 5: loss 0.1044 | dev strict F1 0.5256 (P 0.552 R 0.656) | 48s
epoch 6: loss 0.0913 | dev strict F1 0.5087 (P 0.535 R 0.654) | 48s
epoch 7: loss 0.0828 | dev strict F1 0.5084 (P 0.537 R 0.647) | 47s
epoch 8: loss 0.0779 | dev strict F1 0.5099 (P 0.548 R 0.641) | 47s
   -> macro 0.5256 | micro 0.5998

=== LoRA control (seed 1) ===
trainable params: 0.26M | 1513 windows | 1520

### N15e–N16b · Best model: M1 full FT + bidirectional + InLegalNER

**N15d result:** with full fine-tuning, external data raises dev macro-F1 from 0.505 to **0.601** (micro 0.608 → 0.679). With LoRA the gain is +0.038 over 3 seeds, positive on every seed. So the benefit of external data grows with the number of trainable parameters.

* **N15e**: test zip for `m1_full_bidir_ext`, plus the per-label comparison with its control.
* **N16**: two more seeds of the same recipe (new Stage A, then Stage B), giving mean ± std and models for a seed ensemble.
* **N16b**: (i) **ensembles**: average token log-probs over models, then word-Viterbi; (ii) **document-level consistency**: a party/person string keeps one label within a document, with header mentions weighted 3×, and exact repeats of a predicted name elsewhere in the document get the same label. Every variant is scored on dev against the single model, and only a variant that clearly beats it (> +0.02) is submitted.

In [45]:
# ============================================================================================
# N15e — Test zip for m1_full_bidir_ext + per-label F1 vs its control (uses N15b/N15c helpers)
# ============================================================================================
OUT = json.load(open(f"{WORK}/stats/p2_ext_results.json"))
print(f"{'label':<10}{'ctrl':>8}{'ext':>8}{'Δ':>8}")
for l in ner.LABELS:
    a, b = OUT["m1_full_bidir_ctrl"]["per_label"][l], OUT["m1_full_bidir_ext"]["per_label"][l]
    print(f"{l:<10}{a:>8.3f}{b:>8.3f}{b - a:>+8.3f}")

m = load_saved("m1_full_bidir_ext", bidir=True, method="full")      # fp16 weights -> fp32 model
print("\ndev re-check (should be about 0.601):", {k: v for k, v in evaluate(m).items() if k != "per_label"})
print(write_submission(m, "m1_full_bidir_ext"))
if "RUNS" in globals():
    RUNS["m1_full_bidir_ext"] = ("m1", "full")
del m; torch.cuda.empty_cache()

label         ctrl     ext       Δ
APP          0.158   0.324  +0.166
RESP         0.151   0.189  +0.038
A.COUNSEL    0.490   0.545  +0.056
R.COUNSEL    0.545   0.714  +0.169
JUDGE        0.727   0.743  +0.016
WIT          0.000   0.000  +0.000
AUTH         0.232   0.324  +0.092
COURT        0.753   0.760  +0.008
STAT         0.818   0.817  -0.000
PREC         0.372   0.424  +0.052
DATE         0.947   0.950  +0.002
CASENO       0.364   0.818  +0.454

dev re-check (should be about 0.601): {'macro': 0.6007, 'micro': 0.6787}
m1_full_bidir_ext: 4105 entities (117/doc) | label mix [('STAT', 920), ('DATE', 733), ('PREC', 662), ('COURT', 619), ('CASENO', 332)]
   reference entity format: [18, 40, 'COURT'] | ours: [18, 40, 'COURT']
/kaggle/working/submissions_p2/p2_m1_full_bidir_ext/p2_m1_full_bidir_ext.zip


In [46]:
# ============================================================================================
# N16 — Seeds 1 and 2 of M1 full FT + bidir + InLegalNER (fresh Stage A, then Stage B per seed)
#   Uses run() from N15d. Saves p2_ckpt/m1_full_bidir_ext_s1.pt and _s2.pt
# ============================================================================================
FT_LR = 5e-5
OUT = json.load(open(f"{WORK}/stats/p2_ext_results.json"))
ner.strict_f1 = strict_f1_macro
try:
    mA, _ = run("FULL+bidir Stage A (ext)", True, "full", FT_LR, seed=1, data=ext_ex, epochs=EP_A)
    SA = {k: v.detach().cpu().clone() for k, v in mA.state_dict().items()}; del mA; torch.cuda.empty_cache()
    for s in (1, 2):
        name = f"m1_full_bidir_ext_s{s}"
        m, OUT[name] = run("FULL+bidir ext->ours", True, "full", FT_LR, seed=s, init_state=SA)
        torch.save({k: v.half() for k, v in m.state_dict().items()}, f"{WORK}/p2_ckpt/{name}.pt")
        json.dump(OUT, open(f"{WORK}/stats/p2_ext_results.json", "w"), indent=1)
        del m; torch.cuda.empty_cache()
finally:
    ner.strict_f1 = _strict

ks = ["m1_full_bidir_ext", "m1_full_bidir_ext_s1", "m1_full_bidir_ext_s2"]
for f in ("macro", "micro"):
    v = np.array([OUT[k][f] for k in ks])
    print(f"full+bidir+ext {f}: {v.mean():.4f} ± {v.std(ddof=1):.4f}   per seed {np.round(v, 4).tolist()}")


=== FULL+bidir Stage A (ext) (seed 1) ===
trainable params: 100.11M | 4232 windows | 1058 steps
epoch 1: loss 0.3766 | dev strict F1 0.3375 (P 0.534 R 0.484) | 151s
epoch 2: loss 0.0786 | dev strict F1 0.3845 (P 0.652 R 0.507) | 158s
   -> macro 0.3845 | micro 0.5702

=== FULL+bidir ext->ours (seed 1) ===
trainable params: 100.11M | 1513 windows | 1520 steps
epoch 1: loss 0.2909 | dev strict F1 0.5609 (P 0.636 R 0.676) | 62s
epoch 2: loss 0.0634 | dev strict F1 0.6253 (P 0.675 R 0.710) | 63s
epoch 3: loss 0.0217 | dev strict F1 0.6032 (P 0.700 R 0.680) | 63s
epoch 4: loss 0.0092 | dev strict F1 0.5969 (P 0.716 R 0.632) | 62s
epoch 5: loss 0.0041 | dev strict F1 0.6007 (P 0.705 R 0.655) | 62s
epoch 6: loss 0.0023 | dev strict F1 0.5994 (P 0.709 R 0.646) | 61s
epoch 7: loss 0.0012 | dev strict F1 0.6005 (P 0.712 R 0.648) | 62s
epoch 8: loss 0.0008 | dev strict F1 0.6064 (P 0.712 R 0.651) | 61s
   -> macro 0.6253 | micro 0.6921

=== FULL+bidir ext->ours (seed 2) ===
trainable params: 100

In [47]:
# ============================================================================================
# N16b — Ensembles (mean log-probs -> word-Viterbi) x document-level consistency post-processing
#   Every combination is scored on dev; zips are written for the single best one and for the
#   plain 3-seed ensemble.
# ============================================================================================
import re, zipfile
from collections import defaultdict

# ---- 1. which saved models we can load ---------------------------------------------------------
LOCAL = {"m1_full_bidir_ext": (True, "full"), "m1_full_bidir_ext_s1": (True, "full"),
         "m1_full_bidir_ext_s2": (True, "full"), "m1_lora_cw_ext": (False, "lora")}
def get_model(name):
    if name in LOCAL:
        return load_saved(name, *LOCAL[name]).eval()
    return load_run(name).to(DEV).eval()                           # older runs via N12b's loader

# ---- 2. test docs + cache per-model log-probs for dev and test ---------------------------------
tp = glob.glob(f"{WORK}/data_p2/Phase_2_LNER/Test/*text*.json")[0]
test_docs = [{"id": str(k), "text": v, "ents": []} for k, v in json.load(open(tp)).items()]
ENC.update(pm.encode_all(test_docs, tk))
CAND = ["m1_full_bidir_ext", "m1_full_bidir_ext_s1", "m1_full_bidir_ext_s2",
        "m1_lora_cw_ext", "nokd_student_m1x6", "m4_adapter"]
LP = {}
for name in CAND:
    try:
        m = get_model(name)
    except Exception as e:
        print(f"skip {name}: {e}"); continue
    LP[name] = {d["id"]: pm.predict_logits(m, ENC[d["id"]][0], device=DEV).numpy()
                for d in dev_docs + test_docs}
    del m; torch.cuda.empty_cache(); print("cached", name)

def decode(names, docs):
    """Average log-probs over `names`, then word-consistent Viterbi."""
    out = {}
    for d in docs:
        lp = np.mean([LP[n][d["id"]] for n in names], 0)
        spans = ENC[d["id"]][1]
        out[d["id"]] = ner.tags_to_entities(ner.viterbi_words(lp, spans, d["text"]), spans)
    return out

# ---- 3. document-level consistency -------------------------------------------------------------
def consistency(pred, docs, vote_labels, prop_labels, head_chars=1500, min_len=4):
    """R1: a string predicted with any label in vote_labels gets ONE label per doc (majority vote,
           header mentions count 3x).
       R2: exact repeats (case-insensitive, word-bounded) of strings whose label is in prop_labels
           are added where nothing was predicted."""
    text = {d["id"]: d["text"] for d in docs}
    out = {}
    for did, ents in pred.items():
        t = text[did]; key = lambda s, e: re.sub(r"\s+", " ", t[s:e]).lower()
        votes = defaultdict(Counter)
        for s, e, l in ents:
            if l in vote_labels:
                votes[key(s, e)][l] += 3 if s < head_chars else 1
        major = {k: c.most_common(1)[0][0] for k, c in votes.items()}
        new = [[s, e, major.get(key(s, e), l) if l in vote_labels else l] for s, e, l in ents]
        if prop_labels:
            used = np.zeros(len(t) + 1, dtype=bool)
            for s, e, _ in new:
                used[s:e] = True
            for k, lab in major.items():
                if lab not in prop_labels or len(k) < min_len:
                    continue
                pat = r"(?<!\w)" + r"\s+".join(re.escape(w) for w in k.split()) + r"(?!\w)"
                for mt in re.finditer(pat, t, flags=re.I):
                    s, e = mt.span()
                    if not used[s:e].any():
                        new.append([s, e, lab]); used[s:e] = True
        out[did] = sorted(new)
    return out

PARTY  = {"APP", "RESP", "AUTH"}
PERSON = {"APP", "RESP", "AUTH", "JUDGE", "WIT", "A.COUNSEL", "R.COUNSEL"}
POST = {"none":            lambda p, D: p,
        "R1 party":        lambda p, D: consistency(p, D, PARTY, set()),
        "R1+R2 party":     lambda p, D: consistency(p, D, PARTY, {"APP", "RESP"}),
        "R1+R2 person":    lambda p, D: consistency(p, D, PERSON, PERSON - {"AUTH"})}
ENS = {"single s0":       ["m1_full_bidir_ext"],
       "3-seed":          ["m1_full_bidir_ext", "m1_full_bidir_ext_s1", "m1_full_bidir_ext_s2"],
       "3-seed + lora":   ["m1_full_bidir_ext", "m1_full_bidir_ext_s1", "m1_full_bidir_ext_s2", "m1_lora_cw_ext"],
       "3-seed + others": ["m1_full_bidir_ext", "m1_full_bidir_ext_s1", "m1_full_bidir_ext_s2",
                           "nokd_student_m1x6", "m4_adapter"]}
ENS = {k: v for k, v in ENS.items() if all(n in LP for n in v)}

# ---- 4. score every ensemble x post-processing on dev ------------------------------------------
gold = {d["id"]: d["ents"] for d in dev_docs}
TAB = {}
print(f"{'ensemble':<18}" + "".join(f"{p:>16}" for p in POST))
for en, names in ENS.items():
    base = decode(names, dev_docs); row = []
    for pn, f in POST.items():
        r = _strict(f(base, dev_docs), gold); mac = macro_of(r, gold)
        TAB[(en, pn)] = {"macro": mac, "micro": r["f1"], "per_label": r["per_label_f1"]}
        row.append(f"{mac:.4f}/{r['f1']:.3f}")
    print(f"{en:<18}" + "".join(f"{x:>16}" for x in row))
best = max(TAB, key=lambda k: TAB[k]["macro"])
ref = TAB[("single s0", "none")]["macro"]
print(f"\nbest: {best}  macro {TAB[best]['macro']:.4f}  (single model {ref:.4f})")
for l in ["APP", "RESP", "AUTH", "JUDGE", "PREC", "CASENO"]:
    print(f"  {l:<8} single {TAB[('single s0','none')]['per_label'][l]:.3f} -> best {TAB[best]['per_label'][l]:.3f}")
json.dump({f"{a} | {b}": v for (a, b), v in TAB.items()}, open(f"{WORK}/stats/p2_ensemble.json", "w"), indent=1)

# ---- 5. zips: the 3-seed ensemble (safe), and the best combo if it is clearly better (> +0.02) ------
def write_pred_zip(pred, docs, name):
    refj = json.load(open(f"{WORK}/submissions_p2/p2_m1_lora_cw/prediction.json"))
    ex = next(e for v in refj.values() for e in v); text = {d["id"]: d["text"] for d in docs}
    def fmt(doc, s, e, l):
        if isinstance(ex, dict):
            ks = list(ex); return {ks[0]: s, ks[1]: e, ks[2]: l, **({ks[3]: text[doc][s:e]} if len(ks) > 3 else {})}
        return [s, e, l] + ([text[doc][s:e]] if len(ex) > 3 else [])
    out = {d: [fmt(d, s, e, l) for s, e, l in ents] for d, ents in pred.items()}
    folder = f"{WORK}/submissions_p2/p2_{name}"; os.makedirs(folder, exist_ok=True)
    json.dump(out, open(f"{folder}/prediction.json", "w"))
    with zipfile.ZipFile(f"{folder}/p2_{name}.zip", "w", zipfile.ZIP_DEFLATED) as z:
        z.write(f"{folder}/prediction.json", arcname="prediction.json")
    print(f"wrote {name}: {sum(len(v) for v in out.values())} entities")

if "3-seed" in ENS:
    write_pred_zip(decode(ENS["3-seed"], test_docs), test_docs, "ens3_full_bidir_ext")
if TAB[best]["macro"] > max(ref, TAB.get(("3-seed", "none"), {"macro": 0})["macro"]) + 0.02:
    en, pn = best
    nm = "ens_" + re.sub(r"[^a-z0-9]+", "_", f"{en}_{pn}".lower()).strip("_")
    write_pred_zip(POST[pn](decode(ENS[en], test_docs), test_docs), test_docs, nm)
else:
    print("no combination beats the plain models by > 0.02 -> not submitting a post-processed variant")

cached m1_full_bidir_ext
cached m1_full_bidir_ext_s1
cached m1_full_bidir_ext_s2
cached m1_lora_cw_ext
cached nokd_student_m1x6
cached m4_adapter
ensemble                      none        R1 party     R1+R2 party    R1+R2 person
single s0             0.6007/0.679    0.6048/0.681    0.6014/0.660    0.5971/0.659
3-seed                0.6202/0.694    0.6064/0.687    0.6132/0.670    0.6073/0.669
3-seed + lora         0.6350/0.707    0.6206/0.699    0.6297/0.683    0.6341/0.683
3-seed + others       0.6062/0.695    0.6120/0.698    0.6193/0.688    0.6175/0.684

best: ('3-seed + lora', 'none')  macro 0.6350  (single model 0.6007)
  APP      single 0.324 -> best 0.321
  RESP     single 0.189 -> best 0.196
  AUTH     single 0.324 -> best 0.343
  JUDGE    single 0.743 -> best 0.838
  PREC     single 0.424 -> best 0.443
  CASENO   single 0.818 -> best 0.828
wrote ens3_full_bidir_ext: 4206 entities
no combination beats the plain models by > 0.02 -> not submitting a post-processed variant


### N16c · Ensemble submission + two further fine-tuning techniques

**N16/N16b results:**
* Full FT + bidir + InLegalNER: **0.603 ± 0.021** macro over 3 seeds.
* Seed ensemble: 0.620. Seed ensemble + the LoRA-ext model: **0.635** (micro 0.707). The largest gain is on JUDGE (+0.10).
* Document-level consistency rules did **not** help (propagation −0.02 micro). The gold does not label every repeated party mention, so the APP/RESP errors come from mention selection, not from label consistency.

**This cell:**
1. Writes the test zip for the 4-model ensemble.
2. Runs **norm-gain tuning** (only the RMSNorm gains plus the head are trained, about 40k parameters; the analogue of BitFit, since our layers have no biases) and **prompt tuning** (20 trainable soft-prompt vectors plus the head) with the same LoRA-control recipe, completing the comparison of parameter-efficient methods: full / LoRA / LoRMA+ / LoRMAπ / adapters / norm-gain / prompt.

In [48]:
# ============================================================================================
# N16c — (1) zip for the 4-model ensemble  (2) norm-gain and prompt tuning (techniques 6 and 7)
#   Uses N15b/N15d/N16b objects: LP, decode, write_pred_zip, test_docs, run, ours_ex, ...
# ============================================================================================
# ---- 1. ensemble submission (plain mean of log-probs -> word-Viterbi, no post-processing) -----
ENS4 = ["m1_full_bidir_ext", "m1_full_bidir_ext_s1", "m1_full_bidir_ext_s2", "m1_lora_cw_ext"]
write_pred_zip(decode(ENS4, test_docs), test_docs, "ens4_full3_lora_ext")

# ---- 2. two more PEFT techniques, same recipe as the LoRA control (our data only, seed 0) ----
OUT = json.load(open(f"{WORK}/stats/p2_ext_results.json"))
TECH = {                        # name: (method, lr for the method's own parameters)
    "m1_normgain_cw": ("normgain", 1e-3),     # ~40k RMSNorm gains (+ head)
    "m1_prompt_cw":   ("prompt",   1e-2),     # 20 soft-prompt vectors (+ head); prompts need a high lr
}
ner.strict_f1 = strict_f1_macro
try:
    for name, (method, lr) in TECH.items():
        m, OUT[name] = run(f"{method} (ours only)", BIDIR, method, lr, seed=0)
        tr, tot = pp.count(m)
        OUT[name]["trainable_M"] = round(tr / 1e6, 3)
        torch.save({k: v.half() for k, v in m.state_dict().items()}, f"{WORK}/p2_ckpt/{name}.pt")
        if "RUNS" in globals():
            RUNS[name] = ("m1", method)
        write_submission(m, name)                                   # test zip (format as before)
        json.dump(OUT, open(f"{WORK}/stats/p2_ext_results.json", "w"), indent=1)
        del m; torch.cuda.empty_cache()
finally:
    ner.strict_f1 = _strict

# ---- 3. technique comparison on the same recipe (our data only, M1, seed 0) ------------------
print(f"\n{'technique':<22}{'trainable':>11}{'macro':>8}{'micro':>8}")
rows = [("LoRA (control)", "m1_lora_cw_ctrl", 0.26), ("norm-gain", "m1_normgain_cw", None),
        ("prompt tuning", "m1_prompt_cw", None), ("full FT + bidir (ctrl)", "m1_full_bidir_ctrl", 100.1)]
for lab, k, p in rows:
    if k in OUT:
        p = OUT[k].get("trainable_M", p)
        print(f"{lab:<22}{p:>10}M{OUT[k]['macro']:>8.4f}{OUT[k]['micro']:>8.4f}")

wrote ens4_full3_lora_ext: 4257 entities

=== normgain (ours only) (seed 0) ===
trainable params: 0.04M | 1513 windows | 1520 steps
epoch 1: loss 1.0302 | dev strict F1 0.3250 (P 0.361 R 0.553) | 44s
epoch 2: loss 0.2523 | dev strict F1 0.3765 (P 0.443 R 0.562) | 45s
epoch 3: loss 0.2037 | dev strict F1 0.3919 (P 0.449 R 0.583) | 44s
epoch 4: loss 0.1800 | dev strict F1 0.3901 (P 0.451 R 0.584) | 46s
epoch 5: loss 0.1664 | dev strict F1 0.4011 (P 0.472 R 0.589) | 46s
epoch 6: loss 0.1579 | dev strict F1 0.4036 (P 0.479 R 0.587) | 46s
epoch 7: loss 0.1513 | dev strict F1 0.4023 (P 0.475 R 0.586) | 46s
epoch 8: loss 0.1468 | dev strict F1 0.4054 (P 0.479 R 0.587) | 47s
   -> macro 0.4054 | micro 0.5275
m1_normgain_cw: 5496 entities (157/doc) | label mix [('STAT', 1022), ('DATE', 795), ('PREC', 766), ('COURT', 670), ('AUTH', 649)]
   reference entity format: [18, 40, 'COURT'] | ours: [18, 40, 'COURT']

=== prompt (ours only) (seed 0) ===
trainable params: 0.03M | 1513 windows | 1520 steps

### N16d · Final models: retrain on all 70 documents

**Technique comparison** (M1, our data only, same recipe): full FT 0.505 > LoRA 0.495 > prompt tuning 0.439 > norm-gain 0.405 macro. Performance rises with the number of trainable parameters. The two smallest methods over-predict (low precision).

**Final models:** the best recipe (full FT + bidir, InLegalNER Stage A → our data) is retrained on **all 70** documents, for 3 seeds with a fixed 4 epochs (no dev set remains). The test zip is the ensemble of those 3 models plus the LoRA-ext model. Because dev can't score it any more, the check is its agreement (entity-level F1) with the dev-validated `ens4` predictions.

In [52]:
# ============================================================================================
# N16d — Retrain full FT + bidir + ext on ALL 70 docs (3 seeds, fixed 4 epochs) -> final ensemble zip
#   Reuses SA (Stage-A weights left in memory by N16) if present, otherwise recomputes Stage A.
# ============================================================================================
FT_LR, EP_FINAL = 5e-5, 4
all_ex = pm.make_examples(ENC, [d["id"] for d in train_docs + dev_docs], header=0)
print(f"windows: all 70 docs {len(all_ex)} (was {len(ours_ex)} with 60)")

ner.strict_f1 = strict_f1_macro
try:
    if "SA" not in globals():                                       # Stage A again if N16's is gone
        mA, _ = run("FULL+bidir Stage A (ext)", True, "full", FT_LR, seed=1, data=ext_ex, epochs=EP_A)
        SA = {k: v.detach().cpu().clone() for k, v in mA.state_dict().items()}; del mA; torch.cuda.empty_cache()
    for s in (0, 1, 2):
        name = f"m1_full_bidir_ext_all70_s{s}"
        # dev_docs is now INSIDE training, so the per-epoch "dev" score is a training score
        # (it only shows the run is healthy); we keep the model after a fixed EP_FINAL epochs.
        m, _ = run("FULL+bidir ext->ALL 70", True, "full", FT_LR, seed=s, init_state=SA,
                   data=list(all_ex), epochs=EP_FINAL)
        torch.save({k: v.half() for k, v in m.state_dict().items()}, f"{WORK}/p2_ckpt/{name}.pt")
        LP[name] = {d["id"]: pm.predict_logits(m, ENC[d["id"]][0], device=DEV).numpy() for d in test_docs}
        del m; torch.cuda.empty_cache()
finally:
    ner.strict_f1 = _strict

# ---- final ensemble on test + agreement with the dev-validated ens4 --------------------------
FINAL = [f"m1_full_bidir_ext_all70_s{s}" for s in (0, 1, 2)] + ["m1_lora_cw_ext"]
p_final, p_ens4 = decode(FINAL, test_docs), decode(ENS4, test_docs)
agree = _strict(p_final, p_ens4)                                    # treat ens4 as "gold" just for agreement
print(f"\nagreement final vs ens4: F1 {agree['f1']:.3f} (P {agree['precision']:.3f} R {agree['recall']:.3f})")
print(f"entities: final {sum(map(len, p_final.values()))} | ens4 {sum(map(len, p_ens4.values()))}")
print("label mix final:", Counter(e[2] for v in p_final.values() for e in v).most_common())
write_pred_zip(p_final, test_docs, "ens4_final_all70")

windows: all 70 docs 1796 (was 1513 with 60)

=== FULL+bidir ext->ALL 70 (seed 0) ===
trainable params: 100.11M | 1796 windows | 900 steps
epoch 1: loss 0.2286 | dev strict F1 0.8389 (P 0.813 R 0.923) | 68s
epoch 2: loss 0.0528 | dev strict F1 0.9268 (P 0.910 R 0.959) | 71s
epoch 3: loss 0.0176 | dev strict F1 0.9714 (P 0.962 R 0.981) | 72s
epoch 4: loss 0.0058 | dev strict F1 0.9758 (P 0.970 R 0.987) | 71s
   -> macro 0.9758 | micro 0.9784

=== FULL+bidir ext->ALL 70 (seed 1) ===
trainable params: 100.11M | 1796 windows | 900 steps
epoch 1: loss 0.2539 | dev strict F1 0.8587 (P 0.813 R 0.911) | 72s
epoch 2: loss 0.0564 | dev strict F1 0.9303 (P 0.899 R 0.953) | 72s
epoch 3: loss 0.0167 | dev strict F1 0.9681 (P 0.963 R 0.979) | 73s
epoch 4: loss 0.0056 | dev strict F1 0.9777 (P 0.975 R 0.986) | 74s
   -> macro 0.9777 | micro 0.9802

=== FULL+bidir ext->ALL 70 (seed 2) ===
trainable params: 100.11M | 1796 windows | 900 steps
epoch 1: loss 0.2435 | dev strict F1 0.8348 (P 0.785 R 0.918)

## P2.10 · Do M1's own design choices matter downstream?

The Phase 1 ablations (QK-norm on/off, tied/untied embeddings; equal 100M-token budget) are evaluated twice:
1. **Pretraining:** SC validation loss (Cell 48b).
2. **Downstream:** each checkpoint is fine-tuned on L-NER with the *identical* recipe (LoRA r=8 on q and v, class-weighted CE, word-Viterbi, best epoch by dev strict macro-F1). Results are given as mean ± std over 3 seeds.

`abl_base` is the reference. These models are **not** comparable to the 100M-token-plus `m1_dense_100m` used for submissions, only to each other.

In [11]:
# ============================================================================================
# N17 — Downstream (L-NER) evaluation of the M1 ablation checkpoints, 3 seeds each
#   Needs N15b objects (ours_ex, ENC, CW, evaluate, strict_f1_macro, _strict, LR, HEAD_LR, BS, EP_B).
#   Run only after the ablations have finished on GPU 1. About 60 min on GPU 0.
# ============================================================================================
ABL = ["abl_base", "abl_noqk", "abl_untied"]
SEEDS = (0, 1, 2)
RES_ABL_PATH = f"{WORK}/stats/p2_ablation_downstream.json"
RES_ABL = json.load(open(RES_ABL_PATH)) if os.path.exists(RES_ABL_PATH) else {}

# ---- 1. only use runs that have actually finished (a best.pt mid-training would be unfair) ----
ready = []
for a in ABL:
    ck, log = f"{WORK}/checkpoints/{a}/best.pt", f"{WORK}/checkpoints/{a}/stdout.log"
    done = os.path.exists(log) and "COMPLETE" in open(log, errors="ignore").read()
    print(f"{a:<12} best.pt: {os.path.exists(ck)} | finished: {done}")
    if os.path.exists(ck) and done:
        ready.append(a)

# ---- 2. fine-tune each ready checkpoint with the same LoRA recipe ----------------------------
ner.strict_f1 = strict_f1_macro                                   # select epochs by macro-F1
try:
    for a in ready:
        ckpt = f"{WORK}/checkpoints/{a}/best.pt"
        for s in SEEDS:
            key = f"{a}_s{s}"
            if key in RES_ABL:
                continue                                          # resume-safe: skip finished runs
            t0 = time.time()
            m = pp.apply_method(pm.build_tagger(ckpt, bidirectional=BIDIR), METHOD)
            tr, tot = pp.count(m)
            print(f"\n=== {a} | seed {s} | backbone {tot/1e6:.1f}M params, trainable {tr/1e6:.2f}M ===")
            pm.train_tagger(m, ours_ex, dev_docs, ENC, epochs=EP_B, lr=LR, head_lr=HEAD_LR,
                            bs=BS, class_weight=CW, device=DEV, seed=s)
            RES_ABL[key] = {**evaluate(m), "params_M": round(tot / 1e6, 1),
                            "min": round((time.time() - t0) / 60, 1)}
            json.dump(RES_ABL, open(RES_ABL_PATH, "w"), indent=1)  # save after every run
            del m; torch.cuda.empty_cache()
finally:
    ner.strict_f1 = _strict

# ---- 3. table: mean ± std over seeds, and difference from abl_base ---------------------------
def agg(a, f):
    v = np.array([RES_ABL[f"{a}_s{s}"][f] for s in SEEDS if f"{a}_s{s}" in RES_ABL])
    return v.mean(), (v.std(ddof=1) if len(v) > 1 else 0.0), len(v)
print(f"\n{'variant':<12}{'params':>8}{'macro':>18}{'micro':>18}{'Δmacro vs base':>16}")
base = agg("abl_base", "macro")[0] if "abl_base_s0" in RES_ABL else None
for a in ready:
    mm, ms_, n = agg(a, "macro"); mi, mis, _ = agg(a, "micro")
    d = f"{mm - base:+.4f}" if base is not None else "-"
    print(f"{a:<12}{RES_ABL[f'{a}_s0']['params_M']:>7}M{mm:>11.4f} ± {ms_:.3f}{mi:>11.4f} ± {mis:.3f}{d:>16}   (n={n})")
print("\nper-label macro (seed mean):")
for l in ner.LABELS:
    row = [np.mean([RES_ABL[f"{a}_s{s}"]["per_label"][l] for s in SEEDS if f"{a}_s{s}" in RES_ABL]) for a in ready]
    print(f"  {l:<10}" + "".join(f"{x:>10.3f}" for x in row))

abl_base     best.pt: True | finished: True
abl_noqk     best.pt: True | finished: True
abl_untied   best.pt: True | finished: True

=== abl_base | seed 0 | backbone 100.4M params, trainable 0.26M ===
trainable params: 0.26M | 1513 windows | 1520 steps
epoch 1: loss 0.8874 | dev strict F1 0.3330 (P 0.370 R 0.556) | 50s
epoch 2: loss 0.2635 | dev strict F1 0.3568 (P 0.376 R 0.566) | 49s
epoch 3: loss 0.2104 | dev strict F1 0.3935 (P 0.413 R 0.556) | 49s
epoch 4: loss 0.1756 | dev strict F1 0.4000 (P 0.405 R 0.563) | 49s
epoch 5: loss 0.1545 | dev strict F1 0.3933 (P 0.409 R 0.563) | 49s
epoch 6: loss 0.1372 | dev strict F1 0.4003 (P 0.444 R 0.551) | 49s
epoch 7: loss 0.1267 | dev strict F1 0.3936 (P 0.432 R 0.556) | 49s
epoch 8: loss 0.1200 | dev strict F1 0.3889 (P 0.430 R 0.553) | 49s

=== abl_base | seed 1 | backbone 100.4M params, trainable 0.26M ===
trainable params: 0.26M | 1513 windows | 1520 steps
epoch 1: loss 0.9353 | dev strict F1 0.3339 (P 0.374 R 0.525) | 49s
epoch 2: loss 

### P2.10 results · M1 design choices, upstream vs downstream

| variant (100M-token pretraining) | SC val loss | L-NER macro-F1 (3 seeds) | L-NER micro-F1 | fine-tune s/epoch |
|---|---|---|---|---|
| base: QK-norm on, tied embeddings | 3.427 | 0.412 ± 0.010 | 0.489 ± 0.004 | 49 |
| QK-norm **off** | **3.270** | **0.440 ± 0.003** | **0.515 ± 0.004** | **44** |
| untied embeddings (+24.5M params) | 3.293 | 0.421 ± 0.010 | 0.515 ± 0.002 | 49 |

* **QK-norm off is better upstream (−0.157 nats) and downstream (+0.029 macro; every seed above every base seed), and about 10% faster.** Mechanism: RMS-normalised q and k with unit gains, scaled by 1/√64, bound the initial attention logits to about ±8 (typically about ±1). Attention therefore starts diffuse and can only sharpen as the norm gains grow, which a 381-step run doesn't allow. QK-norm's known benefit, stability at high learning rates and in long runs, doesn't pay off at this budget. A learnable temperature or a larger gain initialisation would be the natural fix.
* **Untied embeddings** improve micro-F1 (+0.026) but not macro-F1 (+0.009, within noise), at a cost of 25% more parameters (not parameter-matched).
* The ranking by pretraining loss matches the ranking by downstream micro-F1.
* **Caveats:** one pretraining seed per variant; 100M-token budget; epoch selection on 10 dev documents. The submitted M1 (full budget) keeps QK-norm, and the effect at the full budget is untested.
* **Learning curves** (SC val loss gap to base): QK-norm off: −0.02 (step 50) → −0.22 (step 200, peak) → −0.16 (step 381). Untied: −0.19 → −0.23 (step 100) → −0.13. Both gaps **narrow by 30–40% in the second half of training**, so both choices mainly change *how fast* the model learns at small budgets. This is consistent with keeping QK-norm (for stability) in the full-budget M1.

In [12]:
# ============================================================================================
# N17b — Ablation sanity + learning curves
#   (1) integrity: step + SC loss stored INSIDE each best.pt, file time, duplicate eval lines
#   (2) SC val loss per eval step for base / noqk / untied and the gaps to base
#   If a gap shrinks as steps grow -> that choice mainly changes EARLY learning speed.
# ============================================================================================
import json, os, time, torch
import pandas as pd
ABL = ["abl_base", "abl_noqk", "abl_untied"]
W = "/kaggle/working/checkpoints"

# ---- 1. integrity ------------------------------------------------------------------------------
for a in ABL:
    ck = torch.load(f"{W}/{a}/best.pt", map_location="cpu", weights_only=False)
    sc = ck["val"]["supreme_court"]["loss"]
    mt = time.strftime("%Y-%m-%d %H:%M", time.localtime(os.path.getmtime(f"{W}/{a}/best.pt")))
    n_complete = open(f"{W}/{a}/stdout.log", errors="ignore").read().count("TRAINING COMPLETE")
    print(f"{a:<11} best.pt: step {ck['step']:>3} | SC loss {sc:.4f} | written {mt} | "
          f"'TRAINING COMPLETE' x{n_complete}")
    del ck

# ---- 2. learning curves from eval_log.jsonl ----------------------------------------------------
cols = {}
for a in ABL:
    rows = [json.loads(l) for l in open(f"{W}/{a}/eval_log.jsonl")]
    df = pd.DataFrame({"step": [r["step"] for r in rows],
                       a: [r["supreme_court"]["loss"] for r in rows]})
    dup = df["step"].duplicated().sum()
    if dup:
        print(f"note: {a} has {dup} duplicated eval steps (the double-launch) -> keeping the mean per step")
    cols[a] = df.groupby("step")[a].mean()
T = pd.concat(cols.values(), axis=1)
T["noqk - base"]   = T["abl_noqk"]   - T["abl_base"]
T["untied - base"] = T["abl_untied"] - T["abl_base"]
print("\nSC val loss per eval step (lower is better):")
print(T.round(4).to_string())

# ---- 3. final-step comparison (not just best) --------------------------------------------------
last = T.dropna().iloc[-1]
print(f"\nat step {T.dropna().index[-1]}: base {last['abl_base']:.4f} | noqk {last['abl_noqk']:.4f} "
      f"({last['noqk - base']:+.4f}) | untied {last['abl_untied']:.4f} ({last['untied - base']:+.4f})")

abl_base    best.pt: step 381 | SC loss 3.4269 | written 2026-10-04 10:18 | 'TRAINING COMPLETE' x1
abl_noqk    best.pt: step 381 | SC loss 3.2698 | written 2026-10-04 10:18 | 'TRAINING COMPLETE' x2
abl_untied  best.pt: step 381 | SC loss 3.2925 | written 2026-10-04 10:41 | 'TRAINING COMPLETE' x1
note: abl_untied has 6 duplicated eval steps (the double-launch) -> keeping the mean per step

SC val loss per eval step (lower is better):
      abl_base  abl_noqk  abl_untied  noqk - base  untied - base
step                                                            
50      5.9048    5.8870      5.7119      -0.0178        -0.1929
100     5.2430    5.0639      5.0105      -0.1791        -0.2325
150     4.6893    4.4759      4.4577      -0.2134        -0.2316
200     4.2924    4.0680      4.0576      -0.2244        -0.2348
250     3.9934    3.7883      3.7906      -0.2051        -0.2028
300     3.7652    3.5737      3.5983      -0.1915        -0.1669
350     3.5025    3.3383      3.3622      -

### N17c · Test submissions for the ablation backbones

Each ablation checkpoint (abl_base / abl_noqk / abl_untied; equal 100M-token pretraining budget) is fine-tuned once (seed 0, same LoRA recipe) and submitted. These entries are **not** meant to compete on the leaderboard. They measure the downstream effect of M1's design choices on the 35-document hidden test set, which is more reliable than our 10-document dev set.

In [13]:
# ============================================================================================
# N17c — Fine-tune each ablation backbone once (seed 0) and write a test zip for each
#   Needs R4 objects (tk, ENC, ours_ex, dev_docs, CW, evaluate, strict_f1_macro, _strict, ...).
# ============================================================================================
import zipfile

# ---- test docs -------------------------------------------------------------------------------
tp = glob.glob(f"{WORK}/data_p2/Phase_2_LNER/Test/*text*.json")[0]
test_docs = [{"id": str(k), "text": v, "ents": []} for k, v in json.load(open(tp)).items()]
ENC.update(pm.encode_all(test_docs, tk))

def write_pred_zip(pred, docs, name):
    """Same format as the zips that already scored on Codabench ([start, end, label] per entity)."""
    refj = json.load(open(f"{WORK}/submissions_p2/p2_m1_lora_cw/prediction.json"))
    ex = next(e for v in refj.values() for e in v); text = {d["id"]: d["text"] for d in docs}
    assert set(map(str, refj)) == set(text), "test ids differ from the reference submission"
    def fmt(doc, s, e, l):
        if isinstance(ex, dict):
            ks = list(ex); return {ks[0]: s, ks[1]: e, ks[2]: l, **({ks[3]: text[doc][s:e]} if len(ks) > 3 else {})}
        return [s, e, l] + ([text[doc][s:e]] if len(ex) > 3 else [])
    out = {d: [fmt(d, s, e, l) for s, e, l in ents] for d, ents in pred.items()}
    folder = f"{WORK}/submissions_p2/p2_{name}"; os.makedirs(folder, exist_ok=True)
    json.dump(out, open(f"{folder}/prediction.json", "w"))
    with zipfile.ZipFile(f"{folder}/p2_{name}.zip", "w", zipfile.ZIP_DEFLATED) as z:
        z.write(f"{folder}/prediction.json", arcname="prediction.json")
    print(f"   wrote p2_{name}.zip: {sum(len(v) for v in out.values())} entities")

# ---- train seed 0 per ablation backbone, save, predict test ---------------------------------
ner.strict_f1 = strict_f1_macro                                   # select epoch by dev macro-F1
try:
    for a in ["abl_base", "abl_noqk", "abl_untied"]:
        name = f"{a}_lora_cw"
        m = pp.apply_method(pm.build_tagger(f"{WORK}/checkpoints/{a}/best.pt", bidirectional=BIDIR), METHOD)
        print(f"\n=== {name} ===")
        pm.train_tagger(m, list(ours_ex), dev_docs, ENC, epochs=EP_B, lr=LR, head_lr=HEAD_LR,
                        bs=BS, class_weight=CW, device=DEV, seed=0)
        r = evaluate(m)
        print(f"   dev macro {r['macro']:.4f} | micro {r['micro']:.4f}")
        torch.save({k: v.half() for k, v in m.state_dict().items()}, f"{WORK}/p2_ckpt/{name}.pt")
        write_pred_zip(pm.predict_docs(m, test_docs, ENC, device=DEV), test_docs, name)
        del m; torch.cuda.empty_cache()
finally:
    ner.strict_f1 = _strict


=== abl_base_lora_cw ===
trainable params: 0.26M | 1513 windows | 1520 steps
epoch 1: loss 0.9482 | dev strict F1 0.3373 (P 0.364 R 0.532) | 50s
epoch 2: loss 0.2626 | dev strict F1 0.3566 (P 0.370 R 0.547) | 49s
epoch 3: loss 0.2126 | dev strict F1 0.3735 (P 0.397 R 0.556) | 50s
epoch 4: loss 0.1794 | dev strict F1 0.3913 (P 0.405 R 0.568) | 49s
epoch 5: loss 0.1550 | dev strict F1 0.3995 (P 0.404 R 0.560) | 49s
epoch 6: loss 0.1412 | dev strict F1 0.4069 (P 0.406 R 0.564) | 49s
epoch 7: loss 0.1279 | dev strict F1 0.3980 (P 0.420 R 0.559) | 49s
epoch 8: loss 0.1220 | dev strict F1 0.3949 (P 0.417 R 0.553) | 50s
   dev macro 0.4069 | micro 0.4720
   wrote p2_abl_base_lora_cw.zip: 6205 entities

=== abl_noqk_lora_cw ===
trainable params: 0.26M | 1513 windows | 1520 steps
epoch 1: loss 0.9181 | dev strict F1 0.3463 (P 0.368 R 0.570) | 44s
epoch 2: loss 0.2469 | dev strict F1 0.3829 (P 0.405 R 0.586) | 43s
epoch 3: loss 0.1955 | dev strict F1 0.4169 (P 0.427 R 0.602) | 44s
epoch 4: loss